# Failure Attribution-Guided Agentic Learning for Small Language Models: reproducible experiment notebook

**Research question.** Can a coding agent improve its debugging and recovery decisions by distinguishing pre-existing test failures from regressions introduced by its own edits?

**How to use.** Add the Gemma dataset to the notebook (Add Data), then *Run All*. Nothing else is required: the three original files (`failure_attribution.py`, `attribution_bench.py`, `attribution_prompt_snippet.md`) are embedded verbatim, written to `/kaggle/working/failure_attribution_results/src/`, and hash-checked. If you also upload copies, they are located and compared; mismatches are reported. No GPU, no internet, no API calls, no model downloads.

**What this notebook can and cannot show**
| Part | Evidence | Can support | Cannot support |
|---|---|---|---|
| Synthetic bench (original 10) | author-written scenarios | the code does what it claims on those cases | real-world accuracy |
| Controlled suite (new) | purpose-built git repos, real pytest, outcome-level oracle | classifier behaviour on realistic failure patterns; reproducible defects | Gemma-agent performance; third-party repo results |
| Gemma traces | recorded task outcomes + patches | descriptive/exploratory statistics | causality; regression attribution (no per-edit test output) |
| Patch replay | would need repos, base commits, test envs | -- | not attempted unless prerequisites exist (they are checked) |

**Scientific-integrity rules enforced in code:** no fabricated data (missing inputs produce `NOT EXECUTED`, never substitutes); original classifier preserved and hash-verified; scores only reported if computed in this run; exceptions are raised, not swallowed; headline metrics exclude ambiguous/questionable cases and say so.

In [ ]:
import os, sys, re, json, time, hashlib, shutil, subprocess, tempfile, textwrap, zipfile, io, platform, warnings, importlib, importlib.util, math, fnmatch, traceback
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    HAVE_SNS = True
except Exception:
    sns, HAVE_SNS = None, False
try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda s: s
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 90)
warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------------------------------------------ configuration
OUT = Path("/kaggle/working/failure_attribution_results") if Path("/kaggle/working").exists() else Path("failure_attribution_results")
SRC, FIG, TAB = OUT / "src", OUT / "figures", OUT / "tables"
for p in (OUT, SRC, FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)
DATASET_HINT = Path("/kaggle/input/datasets/anur4ghh/gemma-4-e4b-swe-run-traces-prompts-and-harness/gemma4-swe-agent-traces")
INPUT_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]   # searched for uploaded files / dataset
PREFER_DISCOVERED_SOURCES = False   # True: use uploaded copies of the 3 source files if their hash differs from the embedded ones
RUN_GROUP_OVERRIDE = {}             # optional {run_name: group_name} to define prompt variants yourself
MAX_LOG_LINES = 3_000_000           # cap for log scanning

# ------------------------------------------------------------------ result bookkeeping
RESULTS = {"meta": {}, "executed": {}, "synthetic_bench": {}, "controlled_suite": {}, "comparison": {}, "classifier_review": {},
           "gemma": {}, "patches": {}, "logs": {}, "replay_feasibility": {}, "limitations": []}
def mark(name, status, reason=""):
    RESULTS["executed"][name] = {"status": status, "reason": reason}
    print("[%s] %s%s" % (status, name, (" -- " + reason) if reason else ""))

def sh0(cmd):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60)
        return (r.stdout + r.stderr).strip()
    except Exception as e:
        return "ERROR: %r" % e

GIT_VERSION = sh0("git --version")
PYTEST_VERSION = sh0("%s -m pytest --version" % sys.executable)
GIT_OK = GIT_VERSION.startswith("git version")
PYTEST_OK = "pytest" in PYTEST_VERSION.lower() and "No module" not in PYTEST_VERSION
RESULTS["meta"] = {"python": sys.version.split()[0], "platform": platform.platform(), "git": GIT_VERSION, "pytest": PYTEST_VERSION,
                   "pandas": pd.__version__, "numpy": np.__version__, "seaborn": getattr(sns, "__version__", None),
                   "started": time.strftime("%Y-%m-%d %H:%M:%S")}
print(json.dumps(RESULTS["meta"], indent=2))
print("output dir:", OUT.resolve())
assert GIT_OK, "git is required for the controlled experiments (not found)"
assert PYTEST_OK, "pytest is required (not importable). Run: pip install pytest"


## 1. Original implementation, preserved
The three files you supplied are embedded **verbatim** below, written to disk, imported, and their SHA-256 hashes are re-checked at the end of the run to prove nothing modified them. `FA_ORIG` is the original classifier, `BENCH` the original benchmark module.

In [ ]:
# Verbatim copy of the user's original file (sha256 is verified in the next cell).
ORIG_FA_SRC = r'''"""
failure_attribution.py -- decide whether a test failure is pre-existing
(environment) or caused by the agent's own patch, by comparing test output
captured BEFORE and AFTER the edit and inspecting the patch.

Pure standard library. No third-party imports, no version-specific syntax,
so it runs unchanged on Python 3.8 through the latest release.

Labels
  PASSED            after-run has passing tests and no failures
  PRE_EXISTING_ENV  same environment-type failure before and after; patch not implicated
  NO_EFFECT         same logic failure before and after; patch did not change it (keep fixing)
  PROGRESS          failure changed in a way tied to the patch, same tests (keep fixing)
  REGRESSION_LIKELY new failure that the patch plausibly caused (inspect / fix / revert)
  UNKNOWN           evidence unclear -> keep investigating, do NOT stop early
  NO_SIGNAL         after-output had no recognisable test result

Usage
  from failure_attribution import attribute
  r = attribute(before_text, after_text, diff_text, repo_modules=set(...))
  r["label"], r["action"], r["evidence"]
"""
import re

# ---------- parsing ----------------------------------------------------------

EXC_E_LINE = re.compile(r"^E\s+([A-Za-z_][\w.]*(?:Error|Exception|Exit|Interrupt))\b:?\s*(.*)$")
ASSERT_E_LINE = re.compile(r"^E\s+(assert\s.*)$")
EXC_PLAIN = re.compile(r"^([A-Za-z_][\w.]*(?:Error|Exception))\b:\s*(.*)$")
FRAME_PYTEST = re.compile(r"^([\w./\\-]+\.py):(\d+):")
FRAME_PLAIN = re.compile(r'^\s*File "([^"]+)", line (\d+)')
SUMMARY_ID = re.compile(r"^(FAILED|ERROR)\s+(\S+)")
COUNT = re.compile(r"(\d+)\s+(passed|failed|error|errors)")
NO_MODULE = re.compile(r"No module named ['\"]([\w.]+)['\"]")
CANNOT_IMPORT = re.compile(r"cannot import name ['\"](\w+)['\"]")
FIXTURE_MISSING = re.compile(r"fixture ['\"](\w+)['\"] not found")

ENV_EXC = ("ModuleNotFoundError", "ImportError")


def _norm_msg(msg):
    msg = re.sub(r"0x[0-9a-fA-F]+", "0x?", msg)
    msg = re.sub(r"/tmp/\S+", "/tmp/?", msg)
    msg = re.sub(r"\bline \d+\b", "line ?", msg)
    msg = re.sub(r"\d+\.\d+s", "?s", msg)
    return msg.strip()[:160]


def parse_output(text):
    """Extract structured facts from pytest / python output."""
    info = {"exceptions": [], "frames": [], "failed_ids": set(),
            "passed": 0, "failed": 0, "errors": 0, "has_result": False}
    if not text:
        return info
    for raw in text.splitlines():
        line = raw.rstrip()
        m = EXC_E_LINE.match(line) or EXC_PLAIN.match(line)
        if m:
            info["exceptions"].append((m.group(1).split(".")[-1], _norm_msg(m.group(2))))
        else:
            ma = ASSERT_E_LINE.match(line)
            if ma:
                info["exceptions"].append(("AssertionError", _norm_msg(ma.group(1))))
        m = FRAME_PYTEST.match(line) or FRAME_PLAIN.match(line)
        if m:
            info["frames"].append(m.group(1).replace("\\", "/"))
        m = SUMMARY_ID.match(line)
        if m:
            info["failed_ids"].add(m.group(2).split(" - ")[0])
        for n, kind in COUNT.findall(line):
            info["has_result"] = True
            n = int(n)
            if kind == "passed":
                info["passed"] = max(info["passed"], n)
            elif kind == "failed":
                info["failed"] = max(info["failed"], n)
            else:
                info["errors"] = max(info["errors"], n)
        if "FixtureLookupError" in line or FIXTURE_MISSING.search(line):
            info["exceptions"].append(("FixtureLookupError", _norm_msg(line)))
    if info["exceptions"] or info["failed_ids"]:
        info["has_result"] = True
    return info


def parse_diff(diff_text):
    """Return changed files, added/removed lines and symbols removed by the patch."""
    changed, added, removed = set(), [], []
    for line in (diff_text or "").splitlines():
        if line.startswith("+++ b/"):
            changed.add(line[6:].strip())
        elif line.startswith("--- a/"):
            changed.add(line[6:].strip())
        elif line.startswith("+") and not line.startswith("+++"):
            added.append(line[1:])
        elif line.startswith("-") and not line.startswith("---"):
            removed.append(line[1:])
    sym = re.compile(r"^\s*(?:async\s+def|def|class)\s+(\w+)|^\s*(\w+)\s*=")
    def names(lines):
        out = set()
        for ln in lines:
            m = sym.match(ln)
            if m:
                out.add(m.group(1) or m.group(2))
        return out
    removed_symbols = names(removed) - names(added)
    return {"changed": changed, "added": added, "removed": removed,
            "removed_symbols": removed_symbols}


def _in_changed(frames, changed):
    hits = []
    for fr in frames:
        for ch in changed:
            if fr == ch or fr.endswith("/" + ch) or ch.endswith("/" + fr):
                hits.append(fr)
    return sorted(set(hits))


def _signature(info):
    return (frozenset(info["failed_ids"]), tuple(sorted(set(info["exceptions"]))))


def _is_env(info, repo_modules, diff):
    """Environment-type failure = missing third-party module / fixture that the
    patch did not introduce. A missing module the PATCH ADDED, or a repo-local
    module, is not environmental."""
    added_blob = "\n".join(diff["added"])
    for typ, msg in info["exceptions"]:
        if typ == "FixtureLookupError":
            return True
        if typ in ENV_EXC:
            m = NO_MODULE.search(msg)
            if m:
                root = m.group(1).split(".")[0]
                if root in repo_modules:
                    return False
                if re.search(r"\b" + re.escape(root) + r"\b", added_blob):
                    return False
                return True
    return False


# ---------- decision ---------------------------------------------------------

ACTIONS = {
    "PASSED": "commit",
    "PRE_EXISTING_ENV": "stop fighting the environment; verify spec literals, then commit",
    "NO_EFFECT": "patch did not change the failure; keep fixing (different change)",
    "PROGRESS": "failure moved because of the patch; keep fixing, same test",
    "REGRESSION_LIKELY": "patch likely broke something; inspect diff and traceback, fix or revert",
    "UNKNOWN": "evidence unclear; keep investigating, do not stop early",
    "NO_SIGNAL": "no test result seen; run a targeted check or repro first",
}


def attribute(before_text, after_text, diff_text, repo_modules=None):
    repo_modules = set(repo_modules or [])
    after = parse_output(after_text)
    before = parse_output(before_text) if before_text else None
    diff = parse_diff(diff_text)
    ev = []

    def out(label, conf):
        return {"label": label, "confidence": conf, "action": ACTIONS[label], "evidence": ev}

    if not after["has_result"]:
        ev.append("after-output has no pytest/traceback markers")
        return out("NO_SIGNAL", "low")

    no_fail = after["failed"] == 0 and after["errors"] == 0 and not after["failed_ids"] and not after["exceptions"]
    if after["passed"] > 0 and no_fail:
        ev.append("after-run: %d passed, no failures" % after["passed"])
        return out("PASSED", "medium")  # passing a visible test is NOT proof of correctness

    after_hits = _in_changed(after["frames"], diff["changed"])
    if after_hits:
        ev.append("traceback passes through patched file(s): %s" % ", ".join(after_hits))

    # strongest regression evidence, independent of the baseline
    for typ, msg in after["exceptions"]:
        m = CANNOT_IMPORT.search(msg)
        if m and m.group(1) in diff["removed_symbols"]:
            ev.append("ImportError names '%s', which the patch removed/renamed" % m.group(1))
            return out("REGRESSION_LIKELY", "high")
        m = NO_MODULE.search(msg)
        if m:
            root = m.group(1).split(".")[0]
            if re.search(r"\b" + re.escape(root) + r"\b", "\n".join(diff["added"])):
                ev.append("patch itself added a reference to missing module '%s'" % root)
                return out("REGRESSION_LIKELY", "high")
        if typ == "SyntaxError" or typ == "IndentationError":
            if after_hits:
                ev.append("%s inside a patched file" % typ)
                return out("REGRESSION_LIKELY", "high")

    if before is None or not before["has_result"]:
        ev.append("no usable BEFORE baseline")
        if _is_env(after, repo_modules, diff) and not after_hits:
            ev.append("environment-type error, patch not implicated (baseline missing)")
            return out("UNKNOWN", "low")
        return out("UNKNOWN", "low")

    same = _signature(before) == _signature(after)
    env_after = _is_env(after, repo_modules, diff)
    env_before = _is_env(before, repo_modules, diff)

    if same:
        if env_after and env_before:
            ev.append("identical environment failure before and after the edit")
            return out("PRE_EXISTING_ENV", "high" if not after_hits else "medium")
        ev.append("identical logic failure before and after the edit")
        return out("NO_EFFECT", "medium")

    new_ids = after["failed_ids"] - before["failed_ids"]
    if new_ids:
        ev.append("newly failing tests: %s" % ", ".join(sorted(new_ids)))
        if after_hits:
            return out("REGRESSION_LIKELY", "high")
        return out("REGRESSION_LIKELY", "medium")

    new_exc = set(after["exceptions"]) - set(before["exceptions"])
    if env_before and not env_after and new_exc:
        ev.append("environment error gone, new error: %s" % sorted(new_exc)[0][0])
        return out("PROGRESS", "low")

    if new_exc and after_hits:
        ev.append("new/changed exception %s with patched frames" % sorted(new_exc)[0][0])
        same_tests = before["failed_ids"] == after["failed_ids"]
        return out("PROGRESS" if same_tests else "REGRESSION_LIKELY", "medium")

    if new_exc and before["failed_ids"] == after["failed_ids"] and diff["changed"]:
        ev.append("same failing tests, but observed failure changed after a source edit")
        return out("PROGRESS", "low")

    ev.append("signatures differ but cause not attributable")
    return out("UNKNOWN", "low")


# ---------- the S8-style signature gate, for comparison ---------------------

S8_SANDBOX = re.compile(
    r"httpbin|multipart|no:anyio|ModuleNotFoundError|ImportError|"
    r"fixture .* not found|ERROR collecting|SyntaxError", re.I)


def s8_gate(after_text):
    """Mimics the V10 prompt rule: env/import/collection error -> commit early;
    assertion/traceback -> overtime; clean pass -> submit."""
    after = parse_output(after_text)
    if after["passed"] > 0 and after["failed"] == 0 and after["errors"] == 0:
        return "SUBMIT_PASS"
    if S8_SANDBOX.search(after_text or ""):
        return "COMMIT_AS_SANDBOX"
    return "OVERTIME"
'''


In [ ]:
# Verbatim copy of the user's original file (sha256 is verified in the next cell).
ORIG_BENCH_SRC = r'''"""
attribution_bench.py -- builds small git repos, runs REAL pytest before and
after an edit, and scores failure_attribution.attribute() against the label I
assigned by construction. Also scores the S8-style signature gate.

IMPORTANT: scenarios are written by hand, so this validates that the code
does what it claims. It is NOT evidence about SWE-style benchmark performance.
"""
import os, subprocess, sys, tempfile, textwrap, shutil
from failure_attribution import attribute, s8_gate

LIB_V0 = """
def calculate_total(items):
    return sum(items) - 1   # bug: off by one
"""
TEST_BASIC = """
from lib import calculate_total
def test_total():
    assert calculate_total([1, 2, 3]) == 6
"""
TEST_ENV = """
import package_x
from lib import calculate_total
def test_total():
    assert calculate_total([1, 2, 3]) == 6
"""
TEST_TWO = """
from lib import calculate_total, label
def test_total():
    assert calculate_total([1, 2, 3]) == 6
def test_label():
    assert label(3) == "n=3"
"""
LIB_TWO = """
def calculate_total(items):
    return sum(items) - 1

def label(n):
    return "n=%d" % n
"""

def sh(cmd, cwd):
    return subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True, text=True)

def pytest_out(cwd):
    r = sh("PYTHONDONTWRITEBYTECODE=1 %s -m pytest -q -x --no-header -p no:cacheprovider 2>&1 | head -80" % sys.executable, cwd)
    return r.stdout

def make_repo(files):
    d = tempfile.mkdtemp(prefix="attr_")
    for name, body in files.items():
        with open(os.path.join(d, name), "w") as f:
            f.write(textwrap.dedent(body).lstrip("\n"))
    sh("git init -q && git add -A && git -c user.email=a@b -c user.name=x commit -qm base", d)
    return d

def run_case(files, edit, truth, name, drop_before=False):
    d = make_repo(files)
    before = pytest_out(d)
    edit(d)
    after = pytest_out(d)
    diff = sh("git diff", d).stdout
    mods = {f[:-3] for f in os.listdir(d) if f.endswith(".py")}
    res = attribute(None if drop_before else before, after, diff, repo_modules=mods)
    shutil.rmtree(d, ignore_errors=True)
    return {"name": name, "truth": truth, "pred": res["label"], "conf": res["confidence"],
            "s8": s8_gate(after), "evidence": res["evidence"]}

def sub(path, old, new):
    def f(d):
        p = os.path.join(d, path)
        s = open(p).read()
        assert old in s
        open(p, "w").write(s.replace(old, new))
    return f

CASES = [
    # name, files, edit, truth, s8_should (what a correct stop/continue decision is)
    ("fixed",               {"lib.py": LIB_V0, "test_a.py": TEST_BASIC},
        sub("lib.py", "- 1", ""), "PASSED"),
    ("logic_unchanged",     {"lib.py": LIB_V0, "test_a.py": TEST_BASIC},
        sub("lib.py", "# bug: off by one", "# looked at it"), "NO_EFFECT"),
    ("logic_progress",      {"lib.py": LIB_V0, "test_a.py": TEST_BASIC},
        sub("lib.py", "- 1", "+ 5"), "PROGRESS"),
    ("env_same_unrelated_edit", {"lib.py": LIB_V0, "test_a.py": TEST_ENV},
        sub("lib.py", "- 1", ""), "PRE_EXISTING_ENV"),
    ("env_same_edit_noop",  {"lib.py": LIB_V0, "test_a.py": TEST_ENV},
        sub("lib.py", "# bug: off by one", "# x"), "PRE_EXISTING_ENV"),
    ("rename_breaks_import", {"lib.py": LIB_V0, "test_a.py": TEST_BASIC},
        sub("lib.py", "calculate_total", "compute_total"), "REGRESSION_LIKELY"),
    ("syntax_error_in_patch", {"lib.py": LIB_V0, "test_a.py": TEST_BASIC},
        sub("lib.py", "return sum(items) - 1", "return sum(items) - "), "REGRESSION_LIKELY"),
    ("patch_adds_missing_import", {"lib.py": LIB_V0, "test_a.py": TEST_BASIC},
        sub("lib.py", "def calculate_total", "import nonexistent_pkg\ndef calculate_total"), "REGRESSION_LIKELY"),
    ("patch_breaks_other_test", {"lib.py": LIB_TWO, "test_a.py": TEST_TWO.replace("== 6", "== 5")},
        sub("lib.py", 'return "n=%d" % n', 'return "n=" + n'), "REGRESSION_LIKELY"),
    ("no_baseline_env",     {"lib.py": LIB_V0, "test_a.py": TEST_ENV},
        sub("lib.py", "- 1", ""), "UNKNOWN"),
]
# expected correct action for each truth label: should the agent commit now?
COMMIT_OK = {"PASSED": True, "PRE_EXISTING_ENV": True}

def main():
    rows = []
    for name, files, edit, truth in CASES:
        rows.append(run_case(files, edit, truth, name, drop_before=(name == "no_baseline_env")))
    ok = bad_s8 = 0
    print("%-28s %-18s %-18s %-20s" % ("case", "truth", "attribution", "S8-style gate"))
    for r in rows:
        mark = "OK " if r["pred"] == r["truth"] else "XX "
        s8_commit = r["s8"] in ("SUBMIT_PASS", "COMMIT_AS_SANDBOX")
        should_commit = COMMIT_OK.get(r["truth"], False)
        s8_wrong = s8_commit != should_commit
        ok += r["pred"] == r["truth"]
        bad_s8 += s8_wrong
        print("%s%-25s %-18s %-18s %-20s %s" % (mark, r["name"], r["truth"], r["pred"], r["s8"],
              "<- wrong stop/continue" if s8_wrong else ""))
    print("\nattribution label matches truth: %d/%d" % (ok, len(rows)))
    print("S8-style gate makes wrong commit/continue decision: %d/%d" % (bad_s8, len(rows)))
    return rows

if __name__ == "__main__":
    main()
'''


In [ ]:
# Verbatim copy of the user's original file (sha256 is verified in the next cell).
ORIG_PROMPT_SRC = r'''# BEFORE/AFTER FAILURE ATTRIBUTION (insert in Step 4 of the system prompt)
Capture a baseline BEFORE your first edit, then compare AFTER it. Use the same test command both times, and always set PYTHONDONTWRITEBYTECODE=1 so edited files are really re-read.
- Before editing (one call, with your targeted test): run timeout 150 env PYTHONDONTWRITEBYTECODE=1 python -m pytest TESTFILE -x -q -k KEYWORD > /tmp/before.txt 2>&1 ; tail -15 /tmp/before.txt
- After editing, same command but > /tmp/after.txt, then in one call: grep -E "^E  |^(FAILED|ERROR)" /tmp/before.txt > /tmp/b.sig ; grep -E "^E  |^(FAILED|ERROR)" /tmp/after.txt > /tmp/a.sig ; diff /tmp/b.sig /tmp/a.sig && echo SAME || echo CHANGED ; git diff --stat
Decide from the comparison, not from the error text alone:
1. Tests pass: go to the final check. A passing visible test is not proof; confirm every literal name and message from the issue appears in git diff.
2. SAME and the error is a missing third-party package or missing fixture that your diff did not add: pre-existing environment problem. Do not fight it. Run python -c to import your edited module, confirm the issue literals in git diff, and submit.
3. SAME and it is an assertion or logic error: your edit did not change the behaviour. Make a different change.
4. CHANGED and the new error names something you renamed or removed, or the traceback passes through a file you edited, or a different test now fails: you caused it. Fix it or undo that part with git checkout -- FILE.
5. CHANGED but the cause is unclear, or no baseline exists: treat as UNKNOWN. Keep investigating; never stop early on an unclear failure.
'''


In [ ]:
FILE_INDEX = []
for root in INPUT_ROOTS:
    if not root.exists():
        continue
    for dp, dn, fn in os.walk(root):
        if str(OUT.resolve()) in str(Path(dp).resolve()):
            dn[:] = []; continue
        for f in fn:
            FILE_INDEX.append(os.path.join(dp, f))
        if len(FILE_INDEX) > 800_000:
            break
print("indexed files:", len(FILE_INDEX))

EMBEDDED = {"failure_attribution.py": ORIG_FA_SRC, "attribution_bench.py": ORIG_BENCH_SRC, "attribution_prompt_snippet.md": ORIG_PROMPT_SRC}
def sha(text): return hashlib.sha256(text.encode("utf-8")).hexdigest()
SOURCE_REPORT = {}
for fname, text in EMBEDDED.items():
    found = [p for p in FILE_INDEX if os.path.basename(p) == fname and "failure_attribution_results" not in p]
    entry = {"embedded_sha256": sha(text), "discovered": found[:5], "discovered_match": None, "used": "embedded"}
    for p in found:
        try:
            ok = sha(open(p, encoding="utf-8").read()) == entry["embedded_sha256"]
        except Exception:
            ok = False
        entry["discovered_match"] = ok if entry["discovered_match"] is None else (entry["discovered_match"] and ok)
        if not ok:
            print("WARNING: discovered %s differs from the embedded original: %s" % (fname, p))
            if PREFER_DISCOVERED_SOURCES:
                text = open(p, encoding="utf-8").read(); entry["used"] = "discovered:" + p
    (SRC / fname).write_text(text, encoding="utf-8")
    entry["written_sha256"] = sha(text)
    SOURCE_REPORT[fname] = entry
ORIG_HASHES = {k: v["written_sha256"] for k, v in SOURCE_REPORT.items()}
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
for m in ("failure_attribution", "attribution_bench", "failure_attribution_v2"):
    sys.modules.pop(m, None)
FA_ORIG = importlib.import_module("failure_attribution")
BENCH = importlib.import_module("attribution_bench")
print(pd.DataFrame(SOURCE_REPORT).T[["used", "discovered_match", "embedded_sha256"]])
assert hasattr(FA_ORIG, "attribute") and hasattr(FA_ORIG, "s8_gate") and hasattr(BENCH, "CASES") and hasattr(BENCH, "run_case")
assert callable(BENCH.make_repo) and callable(BENCH.pytest_out)
print("classifier/benchmark interface check passed: attribute(), s8_gate(), CASES, run_case(), make_repo(), pytest_out()")
RESULTS["meta"]["source_hashes"] = ORIG_HASHES
mark("load original classifier + benchmark", "EXECUTED")


## 2. Code review of `failure_attribution.py` and the improved variant
Review findings (each is tested later by a real pytest scenario; the findings table states whether it was **reproduced** or **not reproduced**):

* **F1 false progress** - any changed failure message in the same test gives `PROGRESS`. (This is the problem with the bench's `logic_progress` case.)
* **F2 `-x` masking** - with `-x` the baseline stops at the first failure. If the patch fixes it, the next *pre-existing* failure shows up as a "new failing test" and is labelled `REGRESSION_LIKELY`.
* **F3 missed progress** - the signature ignores line numbers; the failing assert moving later with an identical message is `NO_EFFECT`.
* **F4** - exception-looking lines in captured stdout stop a fully passing run from being `PASSED`.
* **F5 environment misclassification** - a missing *repo-local* module (typical for feature-request tasks) is treated as environmental unless the caller passes complete `repo_modules`; the original bench builds that set from top-level `.py` files only, so packages are invisible.
* **F6 baseline-blind "strong" rules** - a comment that mentions the missing module triggers a *high-confidence* regression even though the baseline fails identically.
* **F7 confidence** - `low/medium/high` are hard-coded constants, not calibrated.
* **F8/F11** - nondeterministic output and flaky tests are not handled (not tested here).
* **F9** - `PASSED` ignores whether the patch edited the tests.
* **F10** - the original `PRE_EXISTING_ENV` action recommends committing although a failing run cannot validate the patch (policy risk). v2 changes the wording; the evaluation scores this label as `VERIFY_INDEPENDENTLY`, never plain `COMMIT`.

Labels express evidence *consistent with* an interpretation, never proof that a patch caused a failure. The improved classifier lives in a **separate module** (`failure_attribution_v2.py`) that imports the untouched original. Every change is logged below.

In [ ]:
# Verbatim copy of the user's original file (sha256 is verified in the next cell).
V2_SRC = r'''"""
failure_attribution_v2.py -- IMPROVED VARIANT of failure_attribution.py.

The original module is imported and left untouched; this file only adds
stricter rules on top of its parsers. Every change is tagged C1..C9 and is
listed in the notebook's change log. Same label set and same return shape as
the original, plus an extra "flags" list.

KNOWN REMAINING WEAKNESSES (verified by scenarios in the notebook):
  W1  a dotted missing module ('pkg.sub') is assumed repo-local even when it is a missing submodule of an
      installed third-party package (environmental) -> labelled NO_EFFECT instead of PRE_EXISTING_ENV.
  W2  if the patch edits test files and the run passes, the label is still PASSED (confidence low, flag
      'test_files_modified'); the label itself is not downgraded.
  W3  with a truncated (-x) baseline the PROGRESS/REGRESSION split relies on a function-name linkage
      heuristic designed on the notebook's development scenarios.
All labels express evidence CONSISTENT WITH an interpretation, never proof of causation.

NOTE: these changes were motivated by failures observed on the notebook's own
hand-written scenarios. Scores on those scenarios are therefore
development-set scores, not evidence of generalisation.
"""
import re
import failure_attribution as orig

ACTIONS_V2 = dict(orig.ACTIONS)
ACTIONS_V2["PRE_EXISTING_ENV"] = ("Stop spending effort on the confirmed environmental issue, verify the "
                                  "implementation independently, and do not treat the patch as validated until "
                                  "appropriate checks pass.")
ACTIONS_V2["PASSED"] = ("visible tests pass; confirm every requirement from the task is covered and that the "
                        "tests were not edited to make them pass, then commit")
ACTIONS_V2["PROGRESS"] = ("failure changed in a way consistent with progress after the patch (not proven); "
                          "keep fixing, same test")
ACTIONS_V2["REGRESSION_LIKELY"] = ("evidence consistent with a regression associated with the patch (not proven); "
                                   "inspect the diff and traceback, then fix or revert if confirmed")
ACTIONS_V2["UNKNOWN"] = ("evidence unclear; re-run BEFORE and AFTER without -x (full run) on the affected "
                         "files, then decide; do not stop early and do not revert on this evidence alone")

SITE_LINE = re.compile(r"^([\w./\\-]+\.py):(\d+): ([A-Za-z_][\w.]*)\s*$")
TRUNCATED = re.compile(r"stopping after \d+ failure|Interrupted:")
NO_TESTS = re.compile(r"no tests ran|collected 0 items", re.I)
HUNK_FUNC = re.compile(r"^@@ .*? @@\s*(?:async\s+def|def|class)\s+(\w+)")
DEF_LINE = re.compile(r"^\s*(?:async\s+def|def|class)\s+(\w+)")
TEST_PATH = re.compile(r"(^|/)(tests?/|test_[^/]*\.py$|[^/]*_test\.py$|conftest\.py$)")


def parse_v2(text):
    """orig.parse_output plus: failure sites, truncation marker, no-tests marker."""
    info = orig.parse_output(text)
    sites = []
    for raw in (text or "").splitlines():
        m = SITE_LINE.match(raw.rstrip())
        if m:
            sites.append((m.group(1).replace("\\", "/"), int(m.group(2)), m.group(3).split(".")[-1]))
    info["sites"] = sites
    info["truncated"] = bool(TRUNCATED.search(text or ""))
    info["no_tests"] = bool(NO_TESTS.search(text or ""))
    info["e_text"] = "\n".join(l for l in (text or "").splitlines() if l.startswith("E ") or SITE_LINE.match(l.rstrip()))
    return info


def touched_names(diff_text):
    """Names of functions/classes whose bodies the patch changes. Tracks the
    enclosing def through hunk header, context and +/- lines, so a changed body
    line is attributed to its nearest preceding def (C5)."""
    names, current = set(), None
    for line in (diff_text or "").splitlines():
        m = HUNK_FUNC.match(line)
        if line.startswith("@@"):
            current = m.group(1) if m else None
            continue
        if line.startswith(("+++", "---", "diff ", "index ")):
            continue
        body = line[1:] if line[:1] in "+- " else line
        dm = DEF_LINE.match(body)
        if dm:
            current = dm.group(1)
            if line[:1] in "+-":
                names.add(current)
        elif line[:1] in "+-" and current and body.strip():
            names.add(current)
    return names


def modules_from_files(paths):
    mods = set()
    for p in paths:
        p = p.replace("\\", "/")
        parts = p.split("/")
        if len(parts) > 1:
            mods.add(parts[0])
        elif p.endswith(".py"):
            mods.add(p[:-3])
    return mods


def _code_added_import(added_lines, root):
    pat = re.compile(r"^\s*(?:import|from)\s+" + re.escape(root) + r"\b")
    return any(pat.match(l) for l in added_lines)


def _env_status(info, repo_modules, diff):
    """(is_env, note). C4: dotted missing module => submodule of an importable
    package => repo-local, not environment. Top-level missing module is env
    unless it is a known repo module or the patch imports it in code."""
    for typ, msg in info["exceptions"]:
        if typ == "FixtureLookupError":
            mm = orig.FIXTURE_MISSING.search(msg)
            if mm and any(mm.group(1) == n for n in diff["removed_symbols"]):
                return False, "fixture removed by patch"
            return True, "missing fixture"
        if typ in orig.ENV_EXC:
            m = orig.NO_MODULE.search(msg)
            if m:
                name = m.group(1)
                root = name.split(".")[0]
                if "." in name:
                    return False, "missing submodule of importable package (repo-local)"
                if root in repo_modules:
                    return False, "missing repo module"
                if _code_added_import(diff["added"], root):
                    return False, "patch imports missing module"
                return True, ("missing third-party module" if repo_modules
                              else "missing top-level module; repo modules unknown")
    return False, ""


def _site_move(before, after, changed):
    """'later' / 'earlier' / None when the same failure moved inside an
    unedited file (execution order => evidence of progress/regression)."""
    bs, as_ = before["sites"], after["sites"]
    if not bs or len(bs) != len(as_):
        return None
    for (bf, bl, be), (af, al, ae) in zip(bs, as_):
        if bf != af or be != ae:
            return None
        if bl != al:
            if any(bf == c or bf.endswith("/" + c) or c.endswith("/" + bf) for c in changed):
                return None
            return "later" if al > bl else "earlier"
    return None


def _mentions_changed(text, changed):
    for c in changed:
        base = c.split("/")[-1]
        if base and re.search(r"\b" + re.escape(base) + r"\b", text or ""):
            return True
    return False


def attribute_v2(before_text, after_text, diff_text, repo_modules=None, repo_files=None):
    repo_modules = set(repo_modules or [])
    if repo_files:
        repo_modules |= modules_from_files(repo_files)
    after = parse_v2(after_text)
    before = parse_v2(before_text) if before_text else None
    diff = orig.parse_diff(diff_text)
    ev, flags = [], []

    def out(label, conf):
        return {"label": label, "confidence": conf, "action": ACTIONS_V2[label],
                "evidence": ev, "flags": flags}

    if not after["has_result"] or (after["no_tests"] and after["passed"] == 0 and after["failed"] == 0):
        ev.append("after-output has no test result (or no tests ran)")
        return out("NO_SIGNAL", "low")

    # C2: PASSED from summary counts / ids only; stray 'XError:' stdout lines are ignored
    if (after["passed"] > 0 and after["failed"] == 0 and after["errors"] == 0
            and not after["failed_ids"] and not after["truncated"]):
        ev.append("after-run: %d passed, no failures (judged from summary counts only)" % after["passed"])
        if any(TEST_PATH.search(c) for c in diff["changed"]):
            flags.append("test_files_modified")
            ev.append("patch modifies test files: a pass is weaker evidence")
            return out("PASSED", "low")
        return out("PASSED", "medium")

    after_hits = orig._in_changed(after["frames"], diff["changed"])
    if after_hits:
        ev.append("traceback passes through patched file(s): %s" % ", ".join(after_hits))
    have_base = before is not None and before["has_result"]
    before_exc = set(before["exceptions"]) if have_base else set()

    # C3: strong rules fire only for exceptions NOT already present in the baseline
    for typ, msg in after["exceptions"]:
        if (typ, msg) in before_exc:
            continue
        m = orig.CANNOT_IMPORT.search(msg)
        if m and m.group(1) in diff["removed_symbols"]:
            ev.append("ImportError names '%s', which the patch removed/renamed (new vs baseline)" % m.group(1))
            return out("REGRESSION_LIKELY", "high")
        m = orig.NO_MODULE.search(msg)
        if m:
            name = m.group(1)
            root = name.split(".")[0]
            if _code_added_import(diff["added"], root):
                ev.append("patch adds an import of missing module '%s' (new vs baseline)" % root)
                return out("REGRESSION_LIKELY", "high")
        if typ == "FixtureLookupError":
            mm = orig.FIXTURE_MISSING.search(msg)
            if mm and mm.group(1) in diff["removed_symbols"]:
                ev.append("fixture '%s' was removed by the patch (new vs baseline)" % mm.group(1))
                return out("REGRESSION_LIKELY", "high")
        if typ in ("SyntaxError", "IndentationError") and (after_hits or _mentions_changed(after_text, diff["changed"])):
            ev.append("%s inside a patched file (new vs baseline)" % typ)
            return out("REGRESSION_LIKELY", "high")

    if not have_base:
        ev.append("no usable BEFORE baseline")
        return out("UNKNOWN", "low")

    same_sig = orig._signature(before) == orig._signature(after)
    env_a, note_a = _env_status(after, repo_modules, diff)
    env_b, _ = _env_status(before, repo_modules, diff)
    if note_a and "unknown" in note_a:
        flags.append("repo_modules_unknown")
    move = _site_move(before, after, diff["changed"])

    if same_sig:
        if env_a and env_b:
            ev.append("identical environment failure before and after (%s)" % note_a)
            conf = "high" if (not after_hits and "repo_modules_unknown" not in flags) else "medium"
            return out("PRE_EXISTING_ENV", conf)
        if move == "later":
            ev.append("same failure, but it now occurs at a LATER line of an unedited file (earlier statements pass)")
            return out("PROGRESS", "low")
        if move == "earlier":
            ev.append("same failure, but it now occurs at an EARLIER line of an unedited file")
            return out("REGRESSION_LIKELY", "low")
        ev.append("identical logic failure before and after the edit")
        return out("NO_EFFECT", "medium")

    b_ids, a_ids = before["failed_ids"], after["failed_ids"]
    new_ids, fixed_ids = a_ids - b_ids, b_ids - a_ids

    if new_ids:
        ev.append("newly failing tests: %s" % ", ".join(sorted(new_ids)))
        if before["truncated"]:
            flags.append("baseline_truncated")
            if after["passed"] > before["passed"]:
                # C5: new failure sits AFTER the old failing test in run order; the baseline never executed it
                linked = [n for n in touched_names(diff_text) if re.search(r"\b" + re.escape(n) + r"\b", after["e_text"])]
                ev.append("baseline was cut short (-x / collection stop); the new failure was never executed before")
                if linked:
                    ev.append("failure text mentions function(s) the patch touched (a clue, not proof of causation): %s" % ", ".join(sorted(linked)))
                    return out("REGRESSION_LIKELY", "medium")
                ev.append("failure text does not mention anything the patch touched; more consistent with an exposed pre-existing failure than a patch-caused one (not established)")
                return out("PROGRESS", "low")
            if after["passed"] < before["passed"]:
                ev.append("new failure is ordered BEFORE the old stop point, so it passed in the baseline")
                return out("REGRESSION_LIKELY", "high")
            return out("UNKNOWN", "low")
        return out("REGRESSION_LIKELY", "high" if after_hits else "medium")

    if fixed_ids and not new_ids:
        ev.append("failing set shrank; no longer failing: %s" % ", ".join(sorted(fixed_ids)))
        return out("PROGRESS", "medium")

    new_exc = set(after["exceptions"]) - before_exc
    if env_b and not env_a and new_exc:
        ev.append("environment error gone, new error: %s" % sorted(new_exc)[0][0])
        return out("PROGRESS", "low")
    if move == "later":
        ev.append("failure moved to a later line of an unedited file")
        return out("PROGRESS", "low")
    if move == "earlier":
        ev.append("failure moved to an earlier line of an unedited file")
        return out("REGRESSION_LIKELY", "low")
    if new_exc:
        # C1: a changed failure text alone is NOT evidence of progress
        ev.append("failure text changed (%s) but nothing shows it moved closer to passing" % sorted(new_exc)[0][0])
        return out("UNKNOWN", "low")
    ev.append("signatures differ but cause not attributable")
    return out("UNKNOWN", "low")
'''


In [ ]:
(SRC / "failure_attribution_v2.py").write_text(V2_SRC, encoding="utf-8")
FA_V2 = importlib.import_module("failure_attribution_v2")
assert FA_V2.orig is FA_ORIG, "v2 must reuse the untouched original module"
print("improved variant loaded:", FA_V2.__file__)
mark("load improved classifier (v2)", "EXECUTED")


In [ ]:
CHANGE_LOG = pd.DataFrame([
 ("C1", "F1", "PROGRESS now requires evidence (failing set shrank, env error replaced by logic error, or failure site moved later in an unedited file). A changed failure message alone -> UNKNOWN.", "removes false PROGRESS"),
 ("C2", "F4", "PASSED judged from summary counts/ids only; stray 'XError:' stdout lines no longer block it. Test-file edits add flag test_files_modified and lower confidence to low, but the label stays PASSED (known weakness W2).", "removes false non-PASSED; flags test tampering"),
 ("C3", "F6", "'Strong' regression rules (removed symbol, added missing import, fixture removed, SyntaxError) fire only for errors NOT already in the baseline; added-import detection looks at real import statements, not comments.", "removes false REGRESSION when baseline already fails"),
 ("C4", "F5", "Dotted missing module ('pkg.sub') is ASSUMED repo-local, not environment (known weakness W1: wrong for missing submodules of installed packages); top-level missing modules are environment unless repo_modules/diff say otherwise; confidence reduced and flag repo_modules_unknown set when repo_modules is empty. repo_files can be passed to derive modules.", "fewer false PRE_EXISTING_ENV on feature-request tasks"),
 ("C5", "F2", "Truncated baselines (-x stop or collection stop) are detected. A 'new' failing test that sorts after the old stop point was never executed before: REGRESSION only if its failure text names a function the patch touched, else PROGRESS (low). A new failure ordered before the old stop point is a genuine regression (high).", "removes -x masking false regression, at the price of lower-confidence labels"),
 ("C6", "F3", "Failure site (file:line of the exception) is parsed; same failure at a later line of an unedited file -> PROGRESS (low), earlier -> REGRESSION (low).", "detects progress hidden by identical messages"),
 ("C7", "F7", "Confidence is capped: 'high' only for deterministic rules; no calibration claim is made (constants remain heuristics).", "documentation/cap only"),
 ("C8", "F10", "PRE_EXISTING_ENV action: 'Stop spending effort on the confirmed environmental issue, verify the implementation independently, and do not treat the patch as validated until appropriate checks pass.' PASSED/PROGRESS/REGRESSION actions use non-causal wording; UNKNOWN says to re-run without -x. The original module's action strings are untouched.", "action wording only"),
 ("C9", "-", "NO_SIGNAL also for 'no tests ran' / 'collected 0 items' outputs.", "minor"),
], columns=["id", "finding", "change", "intended effect"])
display(CHANGE_LOG)
CHANGE_LOG.to_csv(TAB / "improved_classifier_change_log.csv", index=False)
RESULTS["classifier_review"]["change_log"] = CHANGE_LOG.to_dict("records")
print("""
Development-process disclosure: v2 was written after running the original classifier on the dev scenarios (orig10 + ct_dev), so v2 scores on those are
development scores. The 7 ct_holdout scenarios were written before v2 was run on them and v2 was NOT tuned to them; one development-driven fix
(tracking the enclosing function in touched_names after the ct_fix_one_break_another probe) was made after their first run and changed none of their outputs.
The holdout is still authored by the same person who wrote the classifier, so it is a weak, not an independent, test.""")


## 3. Experimental harness (real git, real pytest, independent oracle)
Each scenario is a small git repo. The harness records the baseline, applies a patch, records the after state and the `git diff`, using the **same command both times** with `PYTHONDONTWRITEBYTECODE=1` as in your prompt. Two output modes are captured for every state:

* `x`: `pytest -q -x` truncated to 80 lines, exactly the original bench / prompt protocol.
* `full`: no `-x`, with JUnit XML.

**Ground truth.** The expected label is authored per scenario, then cross-checked by an **outcome-level oracle** computed from the full JUnit results (per-test pass/fail sets and the real file tree, not the classifier's text heuristics). Scenarios split into **tier 1** (authored = oracle), **tier 2** (authored-only; e.g. progress *inside* a still-failing test, which outcome sets cannot see), **tier 3** (ambiguous or questionable). **The primary result is tier 1 only.** Tier 2 (author-assigned labels, not independently verified) and tier 3 are always reported separately and never pooled into a headline number.

**Controlled-experiment feasibility.** Real third-party repositories at reproducible revisions are not available offline in Kaggle, so the controlled experiment uses purpose-built multi-module repos (package + tests + fixtures + conftest). That is what is run; it is *not* described as real-repo evidence.

In [ ]:
import xml.etree.ElementTree as ET

GIT_ENV = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", GIT_AUTHOR_NAME="x", GIT_AUTHOR_EMAIL="a@b",
               GIT_COMMITTER_NAME="x", GIT_COMMITTER_EMAIL="a@b", GIT_CONFIG_NOSYSTEM="1")

def sh(cmd, cwd, timeout=120):
    return subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True, text=True, timeout=timeout, env=GIT_ENV)

def make_repo_nested(files):
    """Like attribution_bench.make_repo but supports sub-directories."""
    d = tempfile.mkdtemp(prefix="attr2_")
    for name, body in files.items():
        p = os.path.join(d, name)
        os.makedirs(os.path.dirname(p) or d, exist_ok=True)
        with open(p, "w") as f:
            f.write(textwrap.dedent(body).lstrip("\n"))
    r = sh("git init -q && git add -A && git -c user.email=a@b -c user.name=x commit -qm base", d)
    assert r.returncode == 0, "git setup failed: " + r.stderr
    return d

def run_pytest(cwd, extra=(), max_lines=80, junit_path=None):
    """Same flags as the original bench / prompt (-q, no cacheprovider, no bytecode)."""
    cmd = [sys.executable, "-m", "pytest", "-q", "--no-header", "-p", "no:cacheprovider"] + list(extra)
    if junit_path:
        cmd.append("--junitxml=" + junit_path)
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, timeout=180, env=GIT_ENV)
    text = (r.stdout + r.stderr).splitlines()[:max_lines]
    return "\n".join(text) + "\n"

def parse_junit(path):
    out = {}
    if not os.path.exists(path):
        return out
    root = ET.parse(path).getroot()
    for tc in root.iter("testcase"):
        tid = "%s::%s" % (tc.get("classname", ""), tc.get("name", ""))
        status, msg = "passed", ""
        for child in tc:
            if child.tag in ("failure", "error"):
                status, msg = ("failed" if child.tag == "failure" else "error"), (child.get("message", "") + " " + (child.text or ""))
            elif child.tag == "skipped" and status == "passed":
                status = "skipped"
        out[tid] = (status, msg)
    return out

ENV_MSG = re.compile(r"No module named ['\"]([\w.]+)['\"]|fixture ['\"]\w+['\"] not found")

def oracle_label(before, after, repo_roots):
    """Independent, OUTCOME-LEVEL ground truth from full (no -x) JUnit results.
    Uses real per-test outcomes and the real file tree (repo_roots), not the
    classifier's text heuristics. Cannot see progress *inside* a still-failing test."""
    ran_after = [t for t, (s, _) in after.items() if s != "skipped"]
    if not ran_after:
        return "NO_SIGNAL"
    fb = {t for t, (s, _) in before.items() if s in ("failed", "error")}
    fa = {t for t, (s, _) in after.items() if s in ("failed", "error")}
    pb = {t for t, (s, _) in before.items() if s == "passed"}
    pa = {t for t, (s, _) in after.items() if s == "passed"}
    if not fa and pa:
        return "PASSED"
    lost = pb - set(after)
    if (fa & pb) or (lost and fa):
        return "REGRESSION_LIKELY"
    if fa == fb:
        def is_env(msgs):
            for m in msgs:
                mm = ENV_MSG.search(m)
                if not mm:
                    return False
                if mm.group(1):
                    name = mm.group(1)
                    if name.split(".")[0] in repo_roots:     # decided from the real file tree, not from the dot
                        return False
                # fixture-not-found counts as environment
            return True
        if is_env([after[t][1] for t in fa]):
            return "PRE_EXISTING_ENV"
        return "NO_EFFECT"
    if (fb & pa) and not (fa - fb):
        return "PROGRESS"
    return "UNKNOWN"

def apply_ops(d, ops):
    if callable(ops):
        return ops(d)
    for op in ops:
        kind = op[0]
        if kind == "sub":
            p = os.path.join(d, op[1]); s = open(p).read()
            assert op[2] in s, "pattern not found in %s: %r" % (op[1], op[2])
            open(p, "w").write(s.replace(op[2], op[3]))
        elif kind == "append":
            with open(os.path.join(d, op[1]), "a") as f:
                f.write(op[2])
        elif kind == "mv":
            os.makedirs(os.path.dirname(os.path.join(d, op[2])) or d, exist_ok=True)
            os.replace(os.path.join(d, op[1]), os.path.join(d, op[2]))
        elif kind == "write":
            with open(os.path.join(d, op[1]), "w") as f:
                f.write(textwrap.dedent(op[2]).lstrip("\n"))
        else:
            raise ValueError(kind)

def repo_module_sets(d):
    files = sh("git ls-files", d).stdout.split()
    orig_style = {f[:-3] for f in os.listdir(d) if f.endswith(".py")}      # exactly what attribution_bench.py does
    full = FA_V2.modules_from_files(files) | orig_style                        # top-level dirs + top-level .py
    return orig_style, full, files

def exec_instance(inst):
    """inst: dict(name, suite, files, ops, truth, ambiguous, questionable, drop_before, pytest_args, probe, note)"""
    d = make_repo_nested(inst["files"])
    args = list(inst.get("pytest_args", []))
    texts, outcomes = {}, {}
    t0 = time.time()
    try:
        for state in ("before", "after"):
            if state == "after":
                apply_ops(d, inst["ops"])
            texts[(state, "x")] = run_pytest(d, ["-x"] + args)
            jp = os.path.join(d, "..", "junit_%s_%s.xml" % (os.path.basename(d), state))
            texts[(state, "full")] = run_pytest(d, args, junit_path=jp)
            outcomes[state] = parse_junit(jp)
            if os.path.exists(jp):
                os.remove(jp)
        sh("git add -A -N", d)
        diff = sh("git diff", d).stdout
        mods_orig, mods_full, files = repo_module_sets(d)
    finally:
        shutil.rmtree(d, ignore_errors=True)
    orc = oracle_label(outcomes["before"], outcomes["after"], mods_full)
    return dict(inst, texts=texts, diff=diff, mods_orig=mods_orig, mods_full=mods_full,
                oracle=orc, n_before=len(outcomes["before"]), n_after=len(outcomes["after"]),
                runtime_s=time.time() - t0)


Scenario definitions. `ct_dev` was used while designing v2; `ct_holdout` was written afterwards (see the disclosure printed in the change-log cell). Six requested situations are all covered: pre-existing failure unchanged by the patch, patch fixes an existing failure, patch introduces a regression, unrelated edit, environment/dependency failure, ambiguous cases.

In [ ]:
# ---------------------------------------------------------------- scenario definitions
BASE = {
    "conftest.py": "",
    "pkg/__init__.py": "",
    "pkg/core.py": """
        def total(items):
            return sum(items) - 1   # bug: off by one

        def scale(x, k=2):
            return x * k
    """,
    "pkg/util.py": """
        def clamp(x, lo, hi):
            return max(lo, min(hi, x))

        def label(n):
            return "n=%d" % n
    """,
    "tests/test_core.py": """
        from pkg.core import total, scale

        def test_total():
            assert total([1, 2, 3]) == 6

        def test_scale():
            assert scale(3) == 6
    """,
    "tests/test_util.py": """
        from pkg.util import clamp, label

        def test_clamp():
            assert clamp(10, 0, 5) == 5

        def test_label():
            assert label(3) == "n=3"
    """,
}

def variant(**changes):
    d = dict(BASE)
    d.update(changes)
    return d

FIX_TOTAL = ("sub", "pkg/core.py", "sum(items) - 1   # bug: off by one", "sum(items)")
HEALTHY = variant(**{"pkg/core.py": """
        def total(items):
            return sum(items)

        def scale(x, k=2):
            return x * k
    """})
LABEL_BUG = """
        def clamp(x, lo, hi):
            return max(lo, min(hi, x))

        def label(n):
            return "n=%d" % (n + 1)   # bug
    """
CLAMP_BUG = """
        def clamp(x, lo, hi):
            return min(lo, max(hi, x))   # bug

        def label(n):
            return "n=%d" % n
    """
FLAGS_TEST = """
        from pkg.core import is_even

        def test_flags():
            assert is_even(2)
            assert is_even(4)
            assert is_even(6)
    """

def scen(name, suite, files, ops, truth, **kw):
    d = dict(name=name, suite=suite, files=files, ops=ops, truth=truth, ambiguous=False,
             questionable=False, drop_before=False, pytest_args=[], probe="", note="")
    d.update(kw)
    return d

CT_DEV = [
    scen("ct_unrelated_edit_preexisting_failure", "ct_dev", BASE,
         [("sub", "pkg/util.py", 'return "n=%d" % n', 'return "n=%d" % int(n)')], "NO_EFFECT"),
    scen("ct_fix_existing_failure", "ct_dev", BASE, [FIX_TOTAL], "PASSED"),
    scen("ct_introduce_regression", "ct_dev", HEALTHY,
         [("sub", "pkg/core.py", "return x * k", "return x + k")], "REGRESSION_LIKELY"),
    scen("ct_fix_one_break_another", "ct_dev", BASE,
         [FIX_TOTAL, ("sub", "pkg/util.py", "return max(lo, min(hi, x))", "return min(lo, max(hi, x))")],
         "REGRESSION_LIKELY", probe="F2",
         note="Fixes the old failure but breaks clamp. Under -x the baseline never ran test_clamp."),
    scen("ct_masked_preexisting_failure", "ct_dev", variant(**{"pkg/util.py": LABEL_BUG}), [FIX_TOTAL],
         "PROGRESS", probe="F2",
         note="Two independent pre-existing failures; patch fixes one. Under -x the second one 'appears' as new."),
    scen("ct_env_missing_dep", "ct_dev",
         variant(**{"tests/test_core.py": "import package_x  # not installed\n" + textwrap.dedent(BASE["tests/test_core.py"]).lstrip("\n")}),
         [FIX_TOTAL], "PRE_EXISTING_ENV"),
    scen("ct_env_missing_fixture", "ct_dev",
         variant(**{"tests/test_core.py": """
            from pkg.core import total, scale

            def test_total(db_conn):
                assert total([1, 2, 3]) == 6

            def test_scale():
                assert scale(3) == 6
         """}), [FIX_TOTAL], "PRE_EXISTING_ENV"),
    scen("ct_env_comment_mentions_module", "ct_dev",
         variant(**{"tests/test_core.py": "import package_x  # not installed\n" + textwrap.dedent(BASE["tests/test_core.py"]).lstrip("\n")}),
         [FIX_TOTAL, ("append", "pkg/core.py", "\n# package_x is an optional extra\n")], "PRE_EXISTING_ENV", probe="F6",
         note="Baseline already fails on package_x; patch only mentions it in a comment."),
    scen("ct_missing_repo_module_feature", "ct_dev",
         variant(**{"tests/test_feature.py": """
            from pkg.feature import frobnicate

            def test_frobnicate():
                assert frobnicate(2) == 4
         """}),
         [("sub", "pkg/util.py", 'return "n=%d" % n', 'return "n=%d" % int(n)')], "NO_EFFECT", probe="F5",
         note="Feature request: test imports a repo-local module that does not exist yet. Not an environment problem."),
    scen("ct_env_missing_submodule_of_installed_pkg", "ct_dev",
         variant(**{"tests/test_core.py": "import email.nonexistent_sub  # installed stdlib package, submodule absent\n" + textwrap.dedent(BASE["tests/test_core.py"]).lstrip("\n")}),
         [FIX_TOTAL], "PRE_EXISTING_ENV", probe="W1",
         note="Dotted missing module whose root is NOT repo-local: environmental. Added AFTER review as a negative probe of v2's dotted-module rule."),
    scen("ct_progress_moved_assert", "ct_dev",
         variant(**{"pkg/core.py": """
            def is_even(n):
                return n < 2
         """, "tests/test_core.py": FLAGS_TEST}),
         [("sub", "pkg/core.py", "return n < 2", "return n <= 2")], "PROGRESS", probe="F3",
         note="Same 'assert False' message, but the failing assert moved from line 4 to line 5."),
    scen("ct_changed_not_closer", "ct_dev", BASE,
         [("sub", "pkg/core.py", "sum(items) - 1   # bug: off by one", "sum(items) + 5")], "UNKNOWN",
         ambiguous=True, probe="F1",
         note="assert 5 == 6 becomes assert 11 == 6: text changed, not demonstrably closer."),
    scen("ct_noisy_pass", "ct_dev",
         variant(**{"tests/test_core.py": """
            from pkg.core import total, scale

            def test_total():
                assert total([1, 2, 3]) == 6

            def test_scale():
                print("ValueError: simulated warning (ignore)")
                assert scale(3) == 6
         """}), [FIX_TOTAL], "PASSED", pytest_args=["-s"], probe="F4",
         note="All tests pass but captured stdout contains a line that looks like an exception."),
    scen("ct_noisy_pass_own_line", "ct_dev",
         variant(**{"tests/test_core.py": """
            from pkg.core import total, scale

            def test_total():
                assert total([1, 2, 3]) == 6

            def test_scale():
                print()
                print("ValueError: simulated warning (ignore)")
                assert scale(3) == 6
         """}), [FIX_TOTAL], "PASSED", pytest_args=["-s"], probe="F4",
         note="Same as ct_noisy_pass but the exception-looking line starts at column 0 (e.g. a logger/stdout line of its own)."),
    scen("ct_weakened_test", "ct_dev", BASE,
         [("sub", "tests/test_core.py", "== 6\n\ndef test_scale", "== 5\n\ndef test_scale")], "UNKNOWN",
         ambiguous=True, probe="F9", note="Patch edits the test's expected value; run passes but proves little."),
    scen("ct_ambiguous_error_type_change", "ct_dev", BASE,
         [("sub", "pkg/core.py", "sum(items) - 1   # bug: off by one", 'sum(items) - "1"')], "UNKNOWN",
         ambiguous=True, note="AssertionError becomes TypeError inside the patched function."),
    scen("ct_rename_breaks_cross_import", "ct_dev", HEALTHY,
         [("sub", "pkg/core.py", "def scale(", "def rescale(")], "REGRESSION_LIKELY"),
    scen("ct_syntax_error_in_package", "ct_dev", HEALTHY,
         [("sub", "pkg/util.py", "return max(lo, min(hi, x))", "return max(lo, min(hi, x)")], "REGRESSION_LIKELY"),
    scen("ct_no_signal", "ct_dev", HEALTHY, [("append", "pkg/core.py", "\n")], "NO_SIGNAL",
         pytest_args=["-k", "zz_no_such_test"], note="Selection matches nothing: no evidence either way."),
]

CT_HOLDOUT = [
    scen("ho_fix_two_of_three_failures", "ct_holdout",
         variant(**{"pkg/util.py": """
            def clamp(x, lo, hi):
                return min(lo, max(hi, x))   # bug

            def label(n):
                return "n=%d" % (n + 1)   # bug
         """}),
         [FIX_TOTAL, ("sub", "pkg/util.py", "return min(lo, max(hi, x))   # bug", "return max(lo, min(hi, x))")],
         "PROGRESS", note="Three independent failures; patch fixes two."),
    scen("ho_noop_whitespace_edit", "ct_holdout", BASE, [("append", "pkg/core.py", "\n\n")], "NO_EFFECT"),
    scen("ho_subpackage_rename_regression", "ct_holdout", HEALTHY, [("mv", "pkg/util.py", "pkg/helpers.py")],
         "REGRESSION_LIKELY", note="Module renamed without updating the importing test."),
    scen("ho_env_edit_adds_unrelated_import", "ct_holdout",
         variant(**{"tests/test_core.py": "import package_x  # not installed\n" + textwrap.dedent(BASE["tests/test_core.py"]).lstrip("\n")}),
         [FIX_TOTAL, ("sub", "pkg/core.py", "def total(", "import json\n\ndef total(")], "PRE_EXISTING_ENV"),
    scen("ho_fixture_removed_regression", "ct_holdout",
         variant(**{"conftest.py": """
            import pytest

            @pytest.fixture
            def sample():
                return [1, 2, 3]
         """, "pkg/core.py": HEALTHY["pkg/core.py"], "tests/test_core.py": """
            from pkg.core import total

            def test_total(sample):
                assert total(sample) == 6
         """}),
         [("write", "conftest.py", "# fixtures removed\n")], "REGRESSION_LIKELY"),
    scen("ho_progress_distinct_messages", "ct_holdout",
         variant(**{"pkg/core.py": """
            def total(items):
                return sum(items) - 1   # bug
         """, "tests/test_core.py": """
            from pkg.core import total

            def test_total():
                assert total([1]) == 1
                assert total([1, 2]) == 3
         """}),
         [("sub", "pkg/core.py", "sum(items) - 1   # bug", "sum(items) - (1 if len(items) > 1 else 0)")],
         "PROGRESS", note="Failure moves from first to second assert with different values/messages."),
    scen("ho_patch_adds_missing_import", "ct_holdout", HEALTHY,
         [("sub", "pkg/core.py", "def total(", "import yaml_missing_pkg\n\ndef total(")], "REGRESSION_LIKELY"),
]


### Execute all scenarios
Includes an equivalence check: the instrumented harness must reproduce the original `BENCH.run_case()` predictions on the original 10 scenarios, otherwise the run stops.

In [ ]:
# Orig-10 scenarios are taken verbatim from attribution_bench.CASES (hand-written; truth = author's label).
ORIG10 = []
for name, files, edit, truth in BENCH.CASES:
    ORIG10.append(scen(name, "orig10", files, edit, truth, drop_before=(name == "no_baseline_env"),
                       questionable=(name == "logic_progress"),
                       note="expected label PROGRESS is unjustified (see audit)" if name == "logic_progress" else ""))
ALL_INSTANCES = ORIG10 + CT_DEV + CT_HOLDOUT
assert len({i["name"] for i in ALL_INSTANCES}) == len(ALL_INSTANCES), "duplicate scenario names"
print("scenarios: orig10=%d  ct_dev=%d  ct_holdout=%d" % (len(ORIG10), len(CT_DEV), len(CT_HOLDOUT)))

EXEC = []
t_all = time.time()
for inst in ALL_INSTANCES:
    try:
        EXEC.append(exec_instance(inst))
    except Exception as e:          # never hide failures that could invalidate results
        print("SCENARIO FAILED:", inst["name"], repr(e)); traceback.print_exc()
        raise
print("executed %d scenarios in %.1fs (each = 4 real pytest runs + git)" % (len(EXEC), time.time() - t_all))

# --- equivalence check: does my harness reproduce the ORIGINAL bench's own run_case() predictions?
equiv = []
for (name, files, edit, truth), mine in zip(BENCH.CASES, EXEC[:len(BENCH.CASES)]):
    t0 = time.time()
    theirs = BENCH.run_case(files, edit, truth, name, drop_before=(name == "no_baseline_env"))
    b = None if mine["drop_before"] else mine["texts"][("before", "x")]
    mine_pred = FA_ORIG.attribute(b, mine["texts"][("after", "x")], mine["diff"], repo_modules=mine["mods_orig"])["label"]
    equiv.append({"case": name, "original_run_case": theirs["pred"], "this_harness": mine_pred, "same": theirs["pred"] == mine_pred,
                  "orig_bench_runtime_s": round(time.time() - t0, 2)})
EQUIV = pd.DataFrame(equiv)
display(EQUIV)
assert EQUIV["same"].all(), "harness does not reproduce the original bench predictions -- investigate before trusting results"
RESULTS["synthetic_bench"]["harness_equivalence"] = EQUIV.to_dict("records")
mark("scenario execution (real git + real pytest)", "EXECUTED", "%d scenarios" % len(EXEC))


### Methods compared on identical cases
* **A_basic** - no attribution: pass -> commit, anything else -> keep fixing.
* **S8_gate** - the original signature gate (`s8_gate`).
* **B_orig** - original classifier with `repo_modules` built like the original bench.
* **B_orig_fullmods** - original classifier given complete repo modules (separates logic flaws from caller-supplied information).
* **C_v2** - improved variant, same inputs as B_orig.
* **D_prompt_rules** - deterministic **emulation** of the five rules in `attribution_prompt_snippet.md`; it is *not* LLM behaviour.

Action mapping: PASSED -> COMMIT, PRE_EXISTING_ENV -> VERIFY_INDEPENDENTLY, NO_EFFECT/PROGRESS -> KEEP_FIXING, REGRESSION_LIKELY -> INSPECT_OR_REVERT, UNKNOWN/NO_SIGNAL -> INVESTIGATE.

In [ ]:
LABELS = ["PASSED", "PRE_EXISTING_ENV", "NO_EFFECT", "PROGRESS", "REGRESSION_LIKELY", "UNKNOWN", "NO_SIGNAL"]
LABEL_TO_ACTION = {"PASSED": "COMMIT", "PRE_EXISTING_ENV": "VERIFY_INDEPENDENTLY", "NO_EFFECT": "KEEP_FIXING", "PROGRESS": "KEEP_FIXING",
                   "REGRESSION_LIKELY": "INSPECT_OR_REVERT", "UNKNOWN": "INVESTIGATE", "NO_SIGNAL": "INVESTIGATE"}
METHODS = ["A_basic", "S8_gate", "B_orig", "B_orig_fullmods", "C_v2", "D_prompt_rules"]
EMITS_LABELS = {"B_orig", "B_orig_fullmods", "C_v2", "D_prompt_rules"}

_SIG = re.compile(r"^E  |^(FAILED|ERROR)")
def prompt_rules_emulation(before, after, diff_text, repo_modules):
    """Deterministic re-implementation of the 5 decision rules in attribution_prompt_snippet.md
    (grep signature -> diff -> rules). This EMULATES the written procedure; it does NOT measure what an LLM
    following the prompt would actually do. The snippet has no 'progress' rule, so PROGRESS is never emitted."""
    if not before:
        return "UNKNOWN"
    pa, d = FA_ORIG.parse_output(after), FA_ORIG.parse_diff(diff_text)
    if pa["passed"] > 0 and pa["failed"] == 0 and pa["errors"] == 0:
        return "PASSED"                                               # rule 1
    bsig = [l for l in before.splitlines() if _SIG.match(l)]
    asig = [l for l in after.splitlines() if _SIG.match(l)]
    if not asig and not pa["has_result"]:
        return "NO_SIGNAL"
    if bsig == asig:                                                  # SAME
        for l in asig:
            m = FA_ORIG.NO_MODULE.search(l)
            if m:
                root = m.group(1).split(".")[0]
                added_import = any(re.match(r"\s*(import|from)\s+" + re.escape(root) + r"\b", x) for x in d["added"])
                if root not in repo_modules and not added_import:
                    return "PRE_EXISTING_ENV"                         # rule 2
            if FA_ORIG.FIXTURE_MISSING.search(l):
                return "PRE_EXISTING_ENV"
        return "NO_EFFECT"                                            # rule 3
    new_lines = set(asig) - set(bsig)                                 # CHANGED -> rule 4 / 5
    removed_named = any((FA_ORIG.CANNOT_IMPORT.search(l) and FA_ORIG.CANNOT_IMPORT.search(l).group(1) in d["removed_symbols"]) for l in new_lines)
    through_edited = bool(FA_ORIG._in_changed(pa["frames"], d["changed"]))
    different_test = bool(pa["failed_ids"] - FA_ORIG.parse_output(before)["failed_ids"])
    return "REGRESSION_LIKELY" if (removed_named or through_edited or different_test) else "UNKNOWN"

def basic_interpretation(after):
    pa = FA_ORIG.parse_output(after)
    return "PASSED" if (pa["passed"] > 0 and pa["failed"] == 0 and pa["errors"] == 0) else "FAILING"

def tier_of(inst):
    if inst["ambiguous"] or inst["questionable"]:
        return 3
    return 1 if inst["oracle"] == inst["truth"] else 2

rows = []
for r in EXEC:
    for mode in ("x", "full"):
        b = None if r["drop_before"] else r["texts"][("before", mode)]
        a = r["texts"][("after", mode)]
        preds = {}
        bas = basic_interpretation(a)
        preds["A_basic"] = (None, None, "COMMIT" if bas == "PASSED" else "KEEP_FIXING", ["after-only: " + bas], [])
        s8 = FA_ORIG.s8_gate(a)
        preds["S8_gate"] = (None, None, "COMMIT" if s8 in ("SUBMIT_PASS", "COMMIT_AS_SANDBOX") else "KEEP_FIXING", [s8], [])
        o = FA_ORIG.attribute(b, a, r["diff"], repo_modules=r["mods_orig"])
        preds["B_orig"] = (o["label"], o["confidence"], LABEL_TO_ACTION[o["label"]], o["evidence"], [])
        o2 = FA_ORIG.attribute(b, a, r["diff"], repo_modules=r["mods_full"])
        preds["B_orig_fullmods"] = (o2["label"], o2["confidence"], LABEL_TO_ACTION[o2["label"]], o2["evidence"], [])
        c = FA_V2.attribute_v2(b, a, r["diff"], repo_modules=r["mods_orig"])
        preds["C_v2"] = (c["label"], c["confidence"], LABEL_TO_ACTION[c["label"]], c["evidence"], c["flags"])
        dl = prompt_rules_emulation(b, a, r["diff"], r["mods_orig"])
        preds["D_prompt_rules"] = (dl, None, LABEL_TO_ACTION[dl], ["emulated prompt rules"], [])
        for m, (lab, conf, act, ev, fl) in preds.items():
            rows.append({"suite": r["suite"], "name": r["name"], "mode": mode, "tier": tier_of(r), "truth": r["truth"], "oracle": r["oracle"],
                         "ambiguous": r["ambiguous"], "questionable": r["questionable"], "probe": r["probe"], "method": m,
                         "pred": lab, "conf": conf, "action_pred": act, "action_truth": LABEL_TO_ACTION[r["truth"]],
                         "evidence": " | ".join(ev), "flags": ",".join(fl)})
PRED = pd.DataFrame(rows)
assert PRED.groupby(["name", "mode", "method"]).size().max() == 1
print("prediction rows:", len(PRED), " (scenarios x 2 modes x %d methods)" % len(METHODS))
INST = pd.DataFrame([{"suite": r["suite"], "name": r["name"], "truth": r["truth"], "oracle": r["oracle"], "tier": tier_of(r),
                      "ambiguous": r["ambiguous"], "probe": r["probe"], "n_tests_before": r["n_before"], "n_tests_after": r["n_after"],
                      "runtime_s": round(r["runtime_s"], 2), "note": r["note"]} for r in EXEC])
display(INST)
print("tier 1 = authored truth confirmed by the independent outcome-level oracle; tier 2 = authored-only (oracle disagrees or cannot see it); tier 3 = ambiguous/questionable")
print(INST["tier"].value_counts().sort_index().to_dict())
INST.to_csv(TAB / "scenario_inventory.csv", index=False); PRED.to_csv(TAB / "predictions_all_methods.csv", index=False)


In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def label_report(y_true, y_pred, labels=LABELS):
    y_true, y_pred = list(y_true), list(y_pred)
    n = len(y_true)
    cm = pd.DataFrame(0, index=labels, columns=labels)
    for t, p in zip(y_true, y_pred):
        cm.loc[t, p] += 1
    per = []
    for l in labels:
        tp = cm.loc[l, l]; fp = cm[l].sum() - tp; fn = cm.loc[l].sum() - tp
        pr = tp / (tp + fp) if tp + fp else float("nan"); rc = tp / (tp + fn) if tp + fn else float("nan")
        f1 = 2 * pr * rc / (pr + rc) if (pr == pr and rc == rc and pr + rc > 0) else (0.0 if (tp + fp + fn) else float("nan"))
        per.append({"label": l, "precision": pr, "recall": rc, "f1": f1, "support": int(cm.loc[l].sum()), "predicted": int(cm[l].sum())})
    per = pd.DataFrame(per)
    acc = sum(t == p for t, p in zip(y_true, y_pred)) / n if n else float("nan")
    macro = per.loc[per["support"] > 0, "f1"].mean()
    try:   # cross-check with scikit-learn when available
        from sklearn.metrics import accuracy_score, confusion_matrix
        assert abs(accuracy_score(y_true, y_pred) - acc) < 1e-9
        assert (confusion_matrix(y_true, y_pred, labels=labels) == cm.values).all()
    except ImportError:
        pass
    return {"n": n, "accuracy": acc, "ci95": wilson(sum(t == p for t, p in zip(y_true, y_pred)), n), "macro_f1_present_classes": macro,
            "per_class": per, "confusion": cm}

def decision_report(df):
    """df: rows of ONE method on ONE case set. Counts are over cases."""
    n = len(df)
    emits = df["pred"].notna().all()
    t, p = df["truth"], df["pred"]
    out = {"n": n,
           "label_accuracy": float((t == p).mean()) if emits and n else None,
           "action_accuracy": float((df["action_truth"] == df["action_pred"]).mean()) if n else None,
           "false_regression": int(((p == "REGRESSION_LIKELY") & (t != "REGRESSION_LIKELY")).sum()) if emits else None,
           "missed_regression": int(((t == "REGRESSION_LIKELY") & (p != "REGRESSION_LIKELY")).sum()) if emits else None,
           "false_env": int(((p == "PRE_EXISTING_ENV") & (t != "PRE_EXISTING_ENV")).sum()) if emits else None,
           "missed_env": int(((t == "PRE_EXISTING_ENV") & (p != "PRE_EXISTING_ENV")).sum()) if emits else None,
           "uncertain_preds": int(p.isin(["UNKNOWN", "NO_SIGNAL"]).sum()) if emits else None,
           "premature_commit": int(((df["action_pred"] == "COMMIT") & (df["action_truth"] != "COMMIT")).sum()),
           "missed_commit": int(((df["action_pred"] != "COMMIT") & (df["action_truth"] == "COMMIT")).sum())}
    return out

CASE_SETS = {
    "PRIMARY: tier 1 (oracle-confirmed)": lambda d: d[d.tier == 1],
    "tier 2 only (authored-only, NOT oracle-confirmed)": lambda d: d[d.tier == 2],
    "tier 1+2 pooled (for reference only)": lambda d: d[d.tier <= 2],
    "orig10_as_authored (all 10)": lambda d: d[d.suite == "orig10"],
    "orig9 (excl. questionable logic_progress)": lambda d: d[(d.suite == "orig10") & ~d.questionable],
    "ct_dev tier 1": lambda d: d[(d.suite == "ct_dev") & (d.tier == 1)],
    "ct_holdout tier 1": lambda d: d[(d.suite == "ct_holdout") & (d.tier == 1)],
    "tier 3 only (ambiguous/questionable)": lambda d: d[d.tier == 3],
}
cmp_rows = []
for set_name, f in CASE_SETS.items():
    for mode in ("x", "full"):
        sub_ = f(PRED[PRED["mode"] == mode])
        for m in METHODS:
            dd = sub_[sub_["method"] == m]
            if len(dd) == 0:
                continue
            rep = decision_report(dd); rep.update({"case_set": set_name, "mode": mode, "method": m})
            cmp_rows.append(rep)
CMP = pd.DataFrame(cmp_rows)[["case_set", "mode", "method", "n", "label_accuracy", "action_accuracy", "false_regression", "missed_regression",
                              "false_env", "missed_env", "uncertain_preds", "premature_commit", "missed_commit"]]
CMP.to_csv(TAB / "method_comparison.csv", index=False)
RESULTS["comparison"]["table"] = CMP.where(CMP.notna(), None).to_dict("records")
mark("metrics + method comparison", "EXECUTED")


## 4. Synthetic benchmark: the original 10 scenarios
Reports accuracy, precision, recall, F1, the confusion matrix and runtime for the original classifier on the original scenarios, then audits the questionable `logic_progress` case. Manually authored scenarios give *preliminary functional validation, not independent proof of real-world performance*; the same author wrote the classifier, the scenarios and the labels, which biases toward agreement.

In [ ]:
def show_label_block(title, d):
    rep = label_report(d["truth"], d["pred"])
    lo, hi = rep["ci95"]
    print("\n=== %s: n=%d  accuracy=%.3f (Wilson 95%% CI %.2f-%.2f)  macro-F1 over present classes=%.3f" % (title, rep["n"], rep["accuracy"], lo, hi, rep["macro_f1_present_classes"]))
    display(rep["per_class"][rep["per_class"][["support", "predicted"]].sum(axis=1) > 0].round(3))
    return rep

b10 = PRED[(PRED.suite == "orig10") & (PRED["mode"] == "x") & (PRED.method == "B_orig")]
print("ORIGINAL classifier on the ORIGINAL 10 scenarios, original protocol (-x -q), labels = author's:")
display(b10[["name", "truth", "pred", "conf", "oracle", "tier", "evidence"]])
rep10 = show_label_block("orig10 as authored (includes questionable logic_progress)", b10)
rep9 = show_label_block("orig9 (questionable case excluded)", b10[~b10.questionable])
print("Confusion matrix, orig10 (rows=truth, cols=pred):"); display(rep10["confusion"].loc[lambda c: c.sum(axis=1) > 0, lambda c: c.sum(axis=0) > 0])
bench_runtime = float(EQUIV["orig_bench_runtime_s"].sum())
print("runtime: original bench run_case x10 = %.1fs; this notebook's instrumented harness (4 pytest runs/scenario) = %.1fs" % (bench_runtime, INST[INST.suite == "orig10"]["runtime_s"].sum()))

print("\nCorrect predictions:", ", ".join(b10[b10.truth == b10.pred].name))
print("Incorrect predictions:", ", ".join(b10[b10.truth != b10.pred].name) or "none")
print("\nHow many of the 10 authored labels does the independent outcome-level oracle confirm?")
print(INST[INST.suite == "orig10"]["tier"].value_counts().sort_index().rename({1: "tier1 confirmed", 2: "tier2 authored-only", 3: "tier3 questionable/ambiguous"}).to_string())
print("Why tier 2: in the original scenarios the baseline ALREADY fails (off-by-one bug), so 'collection error after the edit' cannot be classed as a regression"
      " from outcomes alone, and 'no_baseline_env' is UNKNOWN by construction (the baseline is withheld). These labels are the author's judgement.")

# ---- audit of logic_progress
lp = next(r for r in EXEC if r["name"] == "logic_progress")
grab = lambda t: [l for l in t.splitlines() if l.startswith("E ") or l.startswith("FAILED")]
print("\n=== AUDIT of 'logic_progress' ===")
print("edit: 'return sum(items) - 1'  ->  'return sum(items) + 5'  (test expects 6)")
print("BEFORE failure lines:", grab(lp["texts"][("before", "x")]))
print("AFTER  failure lines:", grab(lp["texts"][("after", "x")]))
print("""Verdict: the output text changed (5 == 6 became 11 == 6) but the computed value moved AWAY from the expected 6, the same single test
still fails at the same assert, and no earlier statement now passes. Nothing here demonstrates progress, so the expected label PROGRESS is NOT justified.
The original classifier labels it PROGRESS (low) only because it treats ANY changed failure message in the same test as progress (finding F1).
Handling: kept visible in the 'orig10 as authored' numbers, EXCLUDED (tier 3) from every headline metric, and replaced by two justified scenarios:
ct_progress_moved_assert (failure moves to a later assert, tier 2) and ct_changed_not_closer (ambiguous, tier 3).""")
RESULTS["synthetic_bench"].update({
    "orig10_accuracy": rep10["accuracy"], "orig10_n": rep10["n"], "orig10_ci95": list(rep10["ci95"]),
    "orig9_accuracy": rep9["accuracy"], "orig9_n": rep9["n"],
    "orig10_per_class": rep10["per_class"].to_dict("records"), "orig10_confusion": rep10["confusion"].to_dict(),
    "orig10_runtime_original_bench_s": bench_runtime,
    "logic_progress_audit": "expected label PROGRESS unjustified: failure value moved away from expected (5->11 vs 6); excluded from headline metrics",
    "orig10_oracle_tiers": INST[INST.suite == "orig10"]["tier"].value_counts().sort_index().to_dict()})
mark("synthetic benchmark evaluation", "EXECUTED")


## 5. Method comparison (identical cases within each table)
Metrics: label accuracy (label-emitting methods only), action accuracy, **false regression** decisions, **missed regressions**, **false/missed environment** calls, uncertain predictions, premature commits and missed commits. Both output modes are shown so the effect of `-x` is visible.

In [ ]:
def show(set_name, mode):
    d = CMP[(CMP.case_set == set_name) & (CMP["mode"] == mode)].drop(columns=["case_set", "mode"]).set_index("method")
    print("\n### %s   [output mode: %s]" % (set_name, "agent protocol -x" if mode == "x" else "full run (no -x)"))
    display(d.round(3))
for s in ["PRIMARY: tier 1 (oracle-confirmed)", "tier 2 only (authored-only, NOT oracle-confirmed)", "ct_dev tier 1", "ct_holdout tier 1", "tier 3 only (ambiguous/questionable)"]:
    for mode in ("x", "full"):
        show(s, mode)
print("\nNotes: identical case sets are used for every method within a table. A_basic and S8_gate emit no 7-class label, so only action-level columns apply to them. They can only choose COMMIT vs KEEP_FIXING, so their action accuracy is capped by design on INSPECT/INVESTIGATE cases; compare them with the others on premature_commit / missed_commit. PRE_EXISTING_ENV maps to VERIFY_INDEPENDENTLY, so committing on an environment failure counts as a premature commit: an environment failure does not show the patch is correct."
      "\nD_prompt_rules is a deterministic EMULATION of the written prompt rules, not measured LLM behaviour. B_orig_fullmods = original classifier given complete repo_modules"
      " (separates classifier logic from caller-supplied information). C_v2 gets the same repo_modules as B_orig.")


## 6. Classifier error analysis
Every mismatch on non-ambiguous cases, error types by method, reproduction status of each review finding, and a confidence-vs-accuracy table (descriptive only; cell sizes are tiny).

In [ ]:
# ---------- error analysis: every non-ambiguous mismatch of the label-emitting classifiers
err = PRED[(PRED.tier <= 2) & PRED.method.isin(["B_orig", "C_v2"]) & (PRED.truth != PRED.pred)].copy()
def err_type(r):
    if r.pred == "REGRESSION_LIKELY": return "false regression"
    if r.truth == "REGRESSION_LIKELY": return "missed regression"
    if r.pred == "PRE_EXISTING_ENV": return "false environment"
    if r.truth == "PRE_EXISTING_ENV": return "missed environment"
    if r.pred == "PROGRESS": return "false progress"
    if r.truth == "PROGRESS": return "missed progress"
    if r.pred in ("UNKNOWN", "NO_SIGNAL"): return "uncertain"
    return "other"
err["error_type"] = err.apply(err_type, axis=1)
ERR = err[["method", "mode", "name", "tier", "truth", "pred", "conf", "error_type", "probe", "evidence"]].sort_values(["method", "mode", "name"])
ERR.to_csv(TAB / "error_analysis.csv", index=False)
print("Mismatches on non-ambiguous cases (tier 1 = oracle-confirmed, tier 2 = authored-only; keep them separate):")
display(ERR)
print("\nError types by method, mode and tier:")
ET = ERR.groupby(["method", "mode", "tier", "error_type"]).size().unstack(fill_value=0)
display(ET)

# ---------- finding status: is each review finding reproduced by a real execution?
FINDINGS = [
 ("F1", "PROGRESS fires on ANY changed failure text in the same test (false progress)", "ct_changed_not_closer / logic_progress"),
 ("F2", "-x truncation: a pre-existing failure that is first visible after the patch is labelled REGRESSION_LIKELY", "ct_masked_preexisting_failure"),
 ("F3", "Failure signature ignores line numbers: failure moving to a later assert is NO_EFFECT (missed progress)", "ct_progress_moved_assert"),
 ("F4", "Exception-looking stdout lines block PASSED because exceptions are parsed from plain lines", "ct_noisy_pass / ct_noisy_pass_own_line"),
 ("F5", "Missing repo-local module is classed as environment unless caller passes complete repo_modules (bench passes top-level .py only)", "ct_missing_repo_module_feature"),
 ("F6", "'Strong' regression rules ignore the baseline: a comment mentioning the missing module -> REGRESSION_LIKELY even when baseline fails identically", "ct_env_comment_mentions_module"),
 ("F9", "PASSED does not check whether the patch edited the tests", "ct_weakened_test"),
]
fr = []
for fid, text, scen_name in FINDINGS:
    names = [s.strip() for s in scen_name.split("/")]
    sub_ = PRED[(PRED.name.isin(names)) & (PRED.method == "B_orig")]
    wrong = sub_[(sub_.truth != sub_.pred) | ((sub_.name == "ct_weakened_test") & (sub_.pred == "PASSED")) | ((sub_.name == "logic_progress") & (sub_.pred == "PROGRESS"))]
    fr.append({"id": fid, "finding": text, "probe_scenario": scen_name,
               "reproduced_by_orig": "YES (%s)" % ", ".join(sorted(set(wrong["mode"] + ":" + wrong.name + "->" + wrong.pred))) if len(wrong) else "NOT reproduced",
               "v2_on_probe": ", ".join(sorted(set(PRED[(PRED.name.isin(names)) & (PRED.method == "C_v2")].apply(lambda r: "%s:%s->%s%s" % (r["mode"], r["name"], r["pred"], (" [" + r["flags"] + "]") if isinstance(r["flags"], str) and r["flags"] else ""), axis=1))))})
FIND = pd.DataFrame(fr)
display(FIND)
FIND.to_csv(TAB / "classifier_review_findings.csv", index=False)
RESULTS["classifier_review"]["findings_reproduction"] = FIND.to_dict("records")
RESULTS["classifier_review"]["not_tested_findings"] = [
 "F7 confidence strings are fixed constants, never calibrated (see calibration table; sample is tiny)",
 "F8 nondeterministic output (random values, timestamps) makes signatures differ -> false CHANGED; not tested here, needs repeated baseline runs",
 "F10 PRE_EXISTING_ENV recommends 'commit' although the failing run cannot validate the patch (policy risk, not a labelling bug)",
 "F11 flaky tests: a newly failing flaky test is labelled REGRESSION_LIKELY; not tested here"]

# ---------- confidence calibration (tiny n -> descriptive only)
cal = PRED[(PRED.tier == 1) & PRED.method.isin(["B_orig", "C_v2"])].copy()
cal["correct"] = cal.truth == cal.pred
CAL = cal.groupby(["method", "mode", "conf"]).agg(n=("correct", "size"), correct=("correct", "sum")).reset_index()
CAL["accuracy"] = (CAL.correct / CAL.n).round(3)
print("\nConfidence tiers vs observed label accuracy (tier 1 cases only). n per cell is small; do not read as calibration:")
display(CAL)
CAL.to_csv(TAB / "confidence_calibration.csv", index=False)
RESULTS["classifier_review"]["calibration"] = CAL.to_dict("records")
RESULTS["classifier_review"]["error_counts"] = ET.reset_index().to_dict("records")
mark("classifier error analysis and calibration", "EXECUTED")

# ---------- remaining weaknesses of the IMPROVED variant, each verified by a real scenario
def _v2(name, mode="x"):
    r = PRED[(PRED.name == name) & (PRED.method == "C_v2") & (PRED["mode"] == mode)].iloc[0]
    return r
w1 = _v2("ct_env_missing_submodule_of_installed_pkg"); w2 = _v2("ct_weakened_test"); w3o = _v2("ct_fix_one_break_another")
V2_WEAK = pd.DataFrame([
 {"id": "W1", "weakness": "dotted missing module is assumed repo-local, even when it is a missing submodule of an installed (third-party/stdlib) package",
  "scenario": "ct_env_missing_submodule_of_installed_pkg", "expected": w1["truth"], "v2_result": w1["pred"] + "/" + str(w1["conf"]),
  "confirmed": bool(w1["pred"] != w1["truth"])},
 {"id": "W2", "weakness": "a patch that edits the tests can still yield PASSED (and a commit-type action); only a low-confidence flag is added",
  "scenario": "ct_weakened_test", "expected": "tests edited: result is weak evidence (tier-3 ambiguous)", "v2_result": w2["pred"] + "/" + str(w2["conf"]) + " flags=" + str(w2["flags"]),
  "confirmed": bool(w2["pred"] == "PASSED")},
 {"id": "W3", "weakness": "under -x the PROGRESS/REGRESSION split depends on a function-name linkage heuristic designed on dev scenarios (not a general solution to truncated baselines)",
  "scenario": "ct_fix_one_break_another / ct_masked_preexisting_failure", "expected": "REGRESSION / PROGRESS", "v2_result": "see comparison tables; correct on these dev scenarios only", "confirmed": True},
])
print("\nRemaining weaknesses of the improved variant (verified by execution where a scenario exists):")
display(V2_WEAK)
V2_WEAK.to_csv(TAB / "v2_remaining_weaknesses.csv", index=False)
RESULTS["classifier_review"]["v2_remaining_weaknesses"] = V2_WEAK.to_dict("records")


## 7. Real Gemma agent-run dataset
**What is observed vs derived.** CSV columns and `result.json` fields are *observed*. Rates, intervals, parsed patch sizes are *derived*. Prompt-variant grouping is a *heuristic* (no structured prompt identifier is assumed). Per-edit test output is *unavailable* unless the logs prove otherwise.

**7.1 Discovery.** Locates the CSV (hint path first, then a search), enumerates folders and zips, and prints counts so mismatches with the expected 23 runs / 454 rows / ~415 patches / 463 results are visible.

In [ ]:
class VF:
    """A file on disk or a member inside a .zip, read lazily."""
    def __init__(self, parts, fs_path=None, zip_path=None, member=None):
        self.parts, self.fs_path, self.zip_path, self.member = tuple(parts), fs_path, zip_path, member
    @property
    def name(self): return self.parts[-1]
    @property
    def display(self): return "/".join(self.parts) + ("   [zip: %s]" % self.zip_path if self.zip_path else "")
    def read_bytes(self):
        if self.zip_path:
            with zipfile.ZipFile(self.zip_path) as z:
                return z.read(self.member)
        return open(self.fs_path, "rb").read()
    def read_text(self): return self.read_bytes().decode("utf-8", errors="replace")
    def iter_lines(self):
        if self.zip_path:
            with zipfile.ZipFile(self.zip_path) as z, z.open(self.member) as f:
                for raw in io.TextIOWrapper(f, encoding="utf-8", errors="replace"):
                    yield raw.rstrip("\n")
        else:
            with open(self.fs_path, encoding="utf-8", errors="replace") as f:
                for raw in f:
                    yield raw.rstrip("\n")

DATA_STATE = {"csv_found": False}
DATA_ROOT, CSV_PATH, DS_FILES = None, None, []
cands = [p for p in FILE_INDEX if os.path.basename(p) == "all_runs_per_task.csv"]
print("candidate all_runs_per_task.csv files:", cands)
if (DATASET_HINT / "all_runs_per_task.csv").exists():
    CSV_PATH = DATASET_HINT / "all_runs_per_task.csv"
elif cands:
    CSV_PATH = Path(max(cands, key=lambda p: os.path.getsize(p)))
if CSV_PATH is None:
    mark("Gemma dataset discovery", "NOT EXECUTED", "all_runs_per_task.csv not found under %s; add the dataset to the notebook (Add Data) or set DATASET_HINT" % [str(r) for r in INPUT_ROOTS])
else:
    DATA_ROOT = CSV_PATH.parent; DATA_STATE["csv_found"] = True
    for dp, dn, fn in os.walk(DATA_ROOT):
        for f in fn:
            full = os.path.join(dp, f); rel = Path(os.path.relpath(full, DATA_ROOT)).parts
            if f.lower().endswith(".zip"):
                try:
                    with zipfile.ZipFile(full) as z:
                        for mem in z.namelist():
                            if not mem.endswith("/"):
                                DS_FILES.append(VF(rel[:-1] + (Path(f).stem,) + Path(mem).parts, zip_path=full, member=mem))
                except zipfile.BadZipFile:
                    print("bad zip:", full)
            else:
                DS_FILES.append(VF(rel, fs_path=full))
    print("dataset root:", DATA_ROOT); print("files visible (incl. zip members):", len(DS_FILES))
    ext = Counter(Path(v.name).suffix.lower() or v.name for v in DS_FILES)
    print("file types:", dict(ext.most_common(12)))
    print("\nfirst levels of the layout:")
    seen = Counter(v.parts[:2] for v in DS_FILES)
    for k, c in list(seen.items())[:12]: print("  ", "/".join(k), "->", c, "files")

def run_task_of(v):
    parts = list(v.parts)
    run = task = None
    if "result_zips" in parts:
        i = parts.index("result_zips")
        if len(parts) > i + 1: run = parts[i + 1]
    elif len(parts) >= 3:
        run = parts[-3]
    if len(parts) >= 2: task = parts[-2]
    if task == run: task = None
    return run, task

PATCH_FILES = [v for v in DS_FILES if v.name == "agent.patch"]
RESULT_FILES = [v for v in DS_FILES if v.name == "result.json"]
LOG_FILES = [v for v in DS_FILES if v.name == "monitor.log" or fnmatch.fnmatch(v.name, "worker_*.log")]
PROMPT_LIKE = [v for v in DS_FILES if re.search(r"prompt|harness|system", v.display, re.I) and Path(v.name).suffix.lower() in (".md", ".txt", ".py", ".json", ".yaml", ".yml", "")]
RUN_DIRS = sorted({run_task_of(v)[0] for v in PATCH_FILES + RESULT_FILES + LOG_FILES if run_task_of(v)[0]})
if DATA_STATE["csv_found"]:
    print("\nrun directories found: %d   agent.patch: %d   result.json: %d   log files: %d   prompt/harness-like files: %d" %
          (len(RUN_DIRS), len(PATCH_FILES), len(RESULT_FILES), len(LOG_FILES), len(PROMPT_LIKE)))
    print("expected per brief: 23 runs, ~415 patches, 463 result.json, 454 CSV rows (differences are reported, not hidden)")
    for v in PROMPT_LIKE[:15]: print("  prompt-like:", v.display)
    RESULTS["gemma"]["discovery"] = {"csv": str(CSV_PATH), "n_run_dirs": len(RUN_DIRS), "n_patches": len(PATCH_FILES), "n_results": len(RESULT_FILES),
                                     "n_logs": len(LOG_FILES), "n_prompt_like": len(PROMPT_LIKE), "run_dirs": RUN_DIRS}
    mark("Gemma dataset discovery", "EXECUTED", "%d runs, %d patches, %d results" % (len(RUN_DIRS), len(PATCH_FILES), len(RESULT_FILES)))


**7.2 Load, validate, merge.** CSV is the anchor; `result.json` is joined on (run, task); duplicates, nulls, key overlap and CSV-vs-JSON agreement are reported. `gold_passes` / `base_fails` are treated as opaque recorded values.

In [ ]:
def to_bool(s):
    m = {"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False, "1.0": True, "0.0": False}
    return s.map(lambda x: x if isinstance(x, bool) else (m.get(str(x).strip().lower(), np.nan) if pd.notna(x) else np.nan))

EXPECTED_CSV_COLS = ["run", "task", "category", "resolved", "verifiable", "tool_calls", "failed_edits", "stop_reason", "elapsed_s", "patch_made",
                     "added_and_deleted_lines_equal_gold"]
CSV_DF = RES_DF = PATCH_DF = MERGED = pd.DataFrame()
QUALITY = {}
if DATA_STATE["csv_found"]:
    CSV_DF = pd.read_csv(CSV_PATH)
    missing_cols = [c for c in EXPECTED_CSV_COLS if c not in CSV_DF.columns]
    extra_cols = [c for c in CSV_DF.columns if c not in EXPECTED_CSV_COLS]
    print("CSV rows:", len(CSV_DF), "| unique runs:", CSV_DF["run"].nunique() if "run" in CSV_DF else "n/a", "| missing expected cols:", missing_cols, "| extra cols:", extra_cols)
    for c in ("run", "task"):
        if c in CSV_DF: CSV_DF[c] = CSV_DF[c].astype(str).str.strip()
    for c in ("resolved", "verifiable", "patch_made", "added_and_deleted_lines_equal_gold"):
        if c in CSV_DF:
            raw_na = CSV_DF[c].isna().sum(); CSV_DF[c] = to_bool(CSV_DF[c])
            unparsed = CSV_DF[c].isna().sum() - raw_na
            if unparsed: print("WARNING: %d unparseable values in %s" % (unparsed, c))
    dup = CSV_DF.duplicated(["run", "task"], keep=False).sum() if {"run", "task"} <= set(CSV_DF.columns) else None
    QUALITY["csv"] = {"rows": len(CSV_DF), "unique_runs": int(CSV_DF["run"].nunique()) if "run" in CSV_DF else None, "unique_tasks": int(CSV_DF["task"].nunique()) if "task" in CSV_DF else None,
                      "duplicate_run_task_rows": None if dup is None else int(dup), "missing_expected_columns": missing_cols,
                      "null_counts": {c: int(v) for c, v in CSV_DF.isna().sum().items() if v}}
    print("duplicate (run,task) rows:", dup); print("null counts:", QUALITY["csv"]["null_counts"])

    # ---- result.json
    recs, bad = [], []
    for v in RESULT_FILES:
        run, task_dir = run_task_of(v)
        try:
            j = json.loads(v.read_text())
        except Exception as e:
            bad.append((v.display, repr(e))); continue
        rec = {"run": run, "task_dir": task_dir, "task": str(j.get("task", task_dir)).strip(), "_src": v.display}
        for k, val in j.items():
            if k == "task": continue
            if isinstance(val, (list, dict)):
                rec[k + "_n"] = len(val); rec[k] = json.dumps(val)[:300]
            else:
                rec[k] = val
        recs.append(rec)
    RES_DF = pd.DataFrame(recs)
    print("result.json parsed: %d ok, %d unparseable" % (len(RES_DF), len(bad)))
    if bad: print("  unparseable examples:", bad[:3])
    if len(RES_DF):
        if len(RESULT_FILES):
            ks = Counter(); [ks.update(json.loads(v.read_text()).keys()) for v in RESULT_FILES[:50]]
            print("result.json keys seen (first 50 files):", sorted(ks))
        QUALITY["result_json"] = {"parsed": len(RES_DF), "unparseable": len(bad), "duplicate_run_task": int(RES_DF.duplicated(["run", "task"]).sum()),
                                  "null_counts": {c: int(v) for c, v in RES_DF.isna().sum().items() if v and not c.startswith("_")}}
        for c in ("resolved", "verifiable"):
            if c in RES_DF: RES_DF[c] = to_bool(RES_DF[c])
        for dcol in ("base_fails", "gold_passes"):
            if dcol in RES_DF:
                print("%s: dtype=%s  sample values=%s   (MEANING NOT VERIFIED: treated as an opaque recorded field)" % (dcol, RES_DF[dcol].dtype, RES_DF[dcol].dropna().astype(str).unique()[:4].tolist()))

    # ---- join keys
    csv_keys = set(zip(CSV_DF.get("run", []), CSV_DF.get("task", [])))
    res_keys = set(zip(RES_DF.get("run", []), RES_DF.get("task", []))) if len(RES_DF) else set()
    print("\nkey overlap (run,task): csv=%d  result.json=%d  both=%d  csv-only=%d  result-only=%d" % (len(csv_keys), len(res_keys), len(csv_keys & res_keys), len(csv_keys - res_keys), len(res_keys - csv_keys)))
    csv_runs, res_runs = set(CSV_DF.get("run", [])), set(RES_DF.get("run", []))
    print("run names: csv=%d result-dirs=%d both=%d" % (len(csv_runs), len(res_runs), len(csv_runs & res_runs)))
    if csv_keys and res_keys and len(csv_keys & res_keys) == 0:
        print("!! NO (run,task) overlap. csv sample:", list(csv_keys)[:3], " result sample:", list(res_keys)[:3], "\n!! Fix run/task derivation before trusting any merged analysis.")
    QUALITY["join"] = {"csv_keys": len(csv_keys), "result_keys": len(res_keys), "both": len(csv_keys & res_keys), "csv_only": len(csv_keys - res_keys), "result_only": len(res_keys - csv_keys),
                       "runs_csv": len(csv_runs), "runs_result_dirs": len(res_runs), "runs_both": len(csv_runs & res_runs)}


**7.3 Patch parsing and merge.** A unified-diff parser validates each hunk against its `@@` counts (self-tested on a known diff), flags empty/malformed patches, and merges patch features into the table.

In [ ]:
HUNK = re.compile(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@")
TESTY = re.compile(r"(^|/)(tests?/|test_[^/]*\.py$|[^/]*_test\.py$|conftest\.py$)")
def parse_patch(text):
    """Unified-diff parser that validates every hunk against its @@ header counts."""
    info = {"empty": not (text or "").strip(), "files": [], "hunks": 0, "added": 0, "deleted": 0, "bad_hunks": 0, "has_header": False}
    if info["empty"]:
        return info
    files, cur = [], None
    def close():
        nonlocal cur
        if cur is not None and not (cur["old"] == cur["exp_old"] and cur["new"] == cur["exp_new"]):
            info["bad_hunks"] += 1
        cur = None
    for line in text.splitlines():
        if cur is not None and cur["old"] >= cur["exp_old"] and cur["new"] >= cur["exp_new"] and not line.startswith("\\"):
            close()                                   # hunk complete by its own header counts
        m = re.match(r"^diff --git a/(.+?) b/(.+)$", line)
        if m and cur is None:
            files.append(m.group(2)); info["has_header"] = True; continue
        if cur is None and line.startswith("+++ "):
            p = line[4:].strip().split("\t")[0]
            if p != "/dev/null":
                p = p[2:] if p.startswith("b/") else p
                if p not in files: files.append(p)
            info["has_header"] = True; continue
        if cur is None and line.startswith("--- "):
            info["has_header"] = True; continue
        h = HUNK.match(line)
        if h:
            close()
            cur = {"exp_old": int(h.group(2)) if h.group(2) is not None else 1, "exp_new": int(h.group(4)) if h.group(4) is not None else 1, "old": 0, "new": 0}
            info["hunks"] += 1; continue
        if line.startswith("\\"):
            continue
        if cur is None:
            if line[:1] in ("+", "-", " ") and line.strip():
                info["bad_hunks"] += 1                # diff body outside any declared hunk
            continue
        if line.startswith("+"): cur["new"] += 1; info["added"] += 1
        elif line.startswith("-"): cur["old"] += 1; info["deleted"] += 1
        else: cur["old"] += 1; cur["new"] += 1
    close()
    info["files"] = files
    info["malformed"] = (not info["has_header"]) or info["hunks"] == 0 or info["bad_hunks"] > 0
    return info

# self-test of the patch parser on a known diff (validation check)
_t = parse_patch("diff --git a/x.py b/x.py\n--- a/x.py\n+++ b/x.py\n@@ -1,2 +1,3 @@\n a\n-b\n+c\n+d\n")
assert (_t["added"], _t["deleted"], _t["hunks"], _t["files"], _t["malformed"]) == (2, 1, 1, ["x.py"], False), _t

if DATA_STATE["csv_found"]:
    prow = []
    for v in PATCH_FILES:
        run, task_dir = run_task_of(v)
        info = parse_patch(v.read_text())
        task = task_dir
        if len(RES_DF):
            m = RES_DF[(RES_DF.run == run) & (RES_DF.task_dir == task_dir)]
            if len(m): task = m.iloc[0]["task"]
        prow.append({"run": run, "task": task, "task_dir": task_dir, "patch_empty": info["empty"], "patch_malformed": bool(info.get("malformed", False)) if not info["empty"] else False,
                     "patch_files": len(info["files"]), "patch_hunks": info["hunks"], "patch_added": info["added"], "patch_deleted": info["deleted"],
                     "patch_changed_lines": info["added"] + info["deleted"], "patch_exts": ";".join(sorted({Path(f).suffix or "(none)" for f in info["files"]})),
                     "patch_touches_tests": any(TESTY.search(f) for f in info["files"]), "patch_file_list": ";".join(info["files"]), "_patch_src": v.display})
    PATCH_DF = pd.DataFrame(prow)
    print("patches parsed:", len(PATCH_DF)); 
    if len(PATCH_DF):
        print("empty: %d  malformed: %d  touching test files: %d  duplicates (run,task_dir): %d" % (PATCH_DF.patch_empty.sum(), PATCH_DF.patch_malformed.sum(), PATCH_DF.patch_touches_tests.sum(), PATCH_DF.duplicated(["run", "task_dir"]).sum()))
    QUALITY["patches"] = {"parsed": len(PATCH_DF), "empty": int(PATCH_DF.patch_empty.sum()) if len(PATCH_DF) else 0, "malformed": int(PATCH_DF.patch_malformed.sum()) if len(PATCH_DF) else 0}

    # ---- merge: CSV is the anchor (one row per recorded task-run)
    M = CSV_DF.copy()
    if len(RES_DF):
        r2 = RES_DF.drop(columns=["task_dir"], errors="ignore").drop_duplicates(["run", "task"], keep="first")
        overlap = [c for c in r2.columns if c in M.columns and c not in ("run", "task")]
        M = M.merge(r2.rename(columns={c: c + "__json" for c in overlap}), on=["run", "task"], how="left", indicator="_json_merge")
        M["has_result_json"] = M["_json_merge"] == "both"; M = M.drop(columns="_json_merge")
        agree = {}
        for c in overlap:
            a, b = M[c], M[c + "__json"]; ok = a.notna() & b.notna()
            if ok.sum():
                try: agree[c] = float((a[ok].astype(str) == b[ok].astype(str)).mean()) if a.dtype == object or b.dtype == object else float(np.isclose(a[ok].astype(float), b[ok].astype(float)).mean())
                except Exception: agree[c] = float((a[ok].astype(str) == b[ok].astype(str)).mean())
        QUALITY["csv_vs_json_agreement"] = agree
        print("\nCSV vs result.json agreement on overlapping columns (fraction equal):", {k: round(v, 3) for k, v in agree.items()})
    else:
        M["has_result_json"] = False
    if len(PATCH_DF):
        p2 = PATCH_DF.drop_duplicates(["run", "task"], keep="first").drop(columns=["task_dir"])
        M = M.merge(p2, on=["run", "task"], how="left", indicator="_p"); M["has_patch_file"] = M["_p"] == "both"; M = M.drop(columns="_p")
    else:
        M["has_patch_file"] = False
    MERGED = M
    print("merged rows:", len(MERGED), "| with result.json:", int(MERGED.has_result_json.sum()), "| with agent.patch:", int(MERGED.has_patch_file.sum()))
    if "patch_made" in MERGED:
        pm = MERGED[MERGED.patch_made.notna()]
        print("patch_made(CSV) vs patch file present & non-empty -> crosstab:")
        display(pd.crosstab(pm["patch_made"], (pm["has_patch_file"] & (pm["patch_empty"] == False)).rename("patch_file_nonempty")))
    QUALITY["merged"] = {"rows": len(MERGED), "with_result_json": int(MERGED.has_result_json.sum()), "with_patch": int(MERGED.has_patch_file.sum())}
    RESULTS["gemma"]["quality"] = QUALITY
    mark("Gemma load / merge", "EXECUTED", "%d merged rows" % len(MERGED))
else:
    mark("Gemma load / merge", "NOT EXECUTED", "dataset not found")


**7.4 Exploratory analysis.** Resolution by run, by common-task subset, by derived group and category; distributions; association with `resolved`. These are correlations: no causal claim is made, and unresolved tasks are never labelled regressions.

In [ ]:
EDA = {}
def has(df, *cols): return len(df) > 0 and all(c in df.columns for c in cols)
def num(df, c): return pd.to_numeric(df[c], errors="coerce") if c in df else pd.Series(dtype=float)
def rate_table(df, by, col="resolved"):
    g = df.dropna(subset=[col]).groupby(by)[col].agg(["size", "sum"]).rename(columns={"size": "n", "sum": "resolved_n"})
    g["rate"] = g.resolved_n / g.n
    ci = g.apply(lambda r: wilson(int(r.resolved_n), int(r.n)), axis=1)
    g["ci_lo"] = [c[0] for c in ci]; g["ci_hi"] = [c[1] for c in ci]
    return g.reset_index()

if not has(MERGED, "run", "resolved"):
    mark("Gemma exploratory analysis", "NOT EXECUTED", "merged data missing or has no 'resolved' column")
else:
    D = MERGED.copy()
    D["resolved"] = D["resolved"].astype(float)
    # ---- group definition (heuristic, derived): config key from result.json fields, or user override
    cfg_cols = [c for c in ("gpu", "time_min", "max_calls") if c in D.columns]
    if RUN_GROUP_OVERRIDE:
        D["group"] = D["run"].map(RUN_GROUP_OVERRIDE).fillna("unassigned"); gsrc = "RUN_GROUP_OVERRIDE"
    elif cfg_cols:
        D["group"] = D[cfg_cols].astype(str).agg("|".join, axis=1); gsrc = "derived from result.json fields " + str(cfg_cols)
    else:
        D["group"] = "unknown"; gsrc = "unavailable"
    print("group definition:", gsrc, "-> prompt-variant identity is NOT recorded in any structured field we found; run name is the only prompt/config identifier.")
    print("NOTE: groups are a heuristic. Prompt text can only be tied to a run if prompt files exist (see prompt-like files above).")

    # ---- per-run
    agg = {"n_tasks": ("task", "size"), "resolved_n": ("resolved", "sum")}
    for c, nm in (("tool_calls", "mean_tool_calls"), ("failed_edits", "mean_failed_edits"), ("elapsed_s", "mean_elapsed_s")):
        if c in D: D[c] = num(D, c); agg[nm] = (c, "mean")
    if "patch_made" in D: D["patch_made"] = D["patch_made"].astype(float); agg["patch_made_rate"] = ("patch_made", "mean")
    if "verifiable" in D: D["verifiable"] = D["verifiable"].astype(float); agg["verifiable_rate"] = ("verifiable", "mean")
    PER_RUN = D.groupby("run").agg(**agg).reset_index()
    PER_RUN["resolution_rate"] = PER_RUN.resolved_n / PER_RUN.n_tasks
    cis = [wilson(int(a), int(b)) for a, b in zip(PER_RUN.resolved_n, PER_RUN.n_tasks)]
    PER_RUN["ci_lo"], PER_RUN["ci_hi"] = [c[0] for c in cis], [c[1] for c in cis]
    PER_RUN = PER_RUN.sort_values("resolution_rate", ascending=False)
    print("\nResolution by run (DIRECTLY OBSERVED counts; rate/CI are derived):"); display(PER_RUN.round(3))
    PER_RUN.to_csv(TAB / "gemma_per_run.csv", index=False)

    # ---- tasks differ across runs -> only compare runs on shared tasks
    cov = D.groupby("task")["run"].nunique()
    thr = max(2, int(np.ceil(0.5 * D["run"].nunique())))
    common = set(cov[cov >= thr].index)
    print("\ntask coverage: %d unique tasks; tasks attempted in >=%d runs: %d" % (len(cov), thr, len(common)))
    if common:
        CT = rate_table(D[D.task.isin(common)], "run"); CT = CT.sort_values("rate", ascending=False)
        print("Resolution on the COMMON task subset (fairer between-run comparison, still confounded by sample size/config):"); display(CT.round(3))
        CT.to_csv(TAB / "gemma_common_tasks_per_run.csv", index=False); EDA["common_task_table"] = CT.to_dict("records")
    GT = rate_table(D, "group"); print("\nResolution by derived group (%s):" % gsrc); display(GT.round(3))
    GT.to_csv(TAB / "gemma_per_group.csv", index=False)
    if "category" in D:
        CAT = rate_table(D, "category"); print("Resolution by category:"); display(CAT.round(3)); CAT.to_csv(TAB / "gemma_per_category.csv", index=False)
    if "stop_reason" in D:
        SR = pd.crosstab(D["stop_reason"], D["resolved"].map({1.0: "resolved", 0.0: "unresolved"})); print("stop_reason x resolved:"); display(SR)

    # ---- distributions
    for c in [x for x in ("tool_calls", "failed_edits", "tool_errors", "edit_errors", "first_edit_call", "elapsed_s", "patch_changed_lines") if x in D]:
        D[c] = num(D, c)
    dist_cols = [c for c in ("tool_calls", "failed_edits", "tool_errors", "edit_errors", "first_edit_call", "patch_changed_lines", "patch_files", "patch_hunks") if c in D and D[c].notna().any()]
    DIST = D[dist_cols].describe(percentiles=[.1, .25, .5, .75, .9]).T; print("\nDistributions (observed or patch-derived):"); display(DIST.round(2))
    DIST.to_csv(TAB / "gemma_distributions.csv")

    # ---- relationships with resolved (descriptive, NOT causal)
    rel = []
    for c in [x for x in ("tool_calls", "failed_edits", "tool_errors", "edit_errors", "first_edit_call", "patch_changed_lines", "patch_files", "patch_hunks", "elapsed_s") if x in D]:
        s = D[[c, "resolved", "run"]].dropna()
        if len(s) < 10 or s[c].nunique() < 2: continue
        rho = s[c].corr(s["resolved"], method="spearman")
        within = [g[c].corr(g["resolved"], method="spearman") for _, g in s.groupby("run") if len(g) >= 8 and g[c].nunique() > 1 and g["resolved"].nunique() > 1]
        rel.append({"metric": c, "n": len(s), "spearman_vs_resolved_pooled": rho, "median_within_run_spearman": float(np.nanmedian(within)) if within else np.nan, "n_runs_used": len(within)})
    REL = pd.DataFrame(rel); print("\nAssociation with `resolved` (Spearman; pooled mixes runs/tasks, within-run controls for run). Correlation only -- no causal claim:"); display(REL.round(3))
    REL.to_csv(TAB / "gemma_relationships.csv", index=False)
    for c in [x for x in ("tool_errors", "patch_changed_lines", "first_edit_call") if x in D and D[x].nunique() >= 4]:
        try:
            D["_bin"] = pd.qcut(D[c], 4, duplicates="drop")
            t = rate_table(D, "_bin"); t.insert(0, "metric", c); print("\nresolution by quartile of %s:" % c); display(t.round(3)); EDA["quartiles_" + c] = t.astype({"_bin": str}).to_dict("records")
        except Exception as e:
            print("quartile table skipped for", c, repr(e))
    D = D.drop(columns=[c for c in ("_bin",) if c in D])
    EDA.update({"per_run": PER_RUN.to_dict("records"), "per_group": GT.to_dict("records"), "group_definition": gsrc, "relationships": REL.to_dict("records"),
                "n_rows": len(D), "n_runs": int(D.run.nunique()), "n_tasks": int(D.task.nunique())})
    MERGED = D
    RESULTS["gemma"]["eda"] = EDA
    mark("Gemma exploratory analysis", "EXECUTED", "%d rows, %d runs" % (len(D), D.run.nunique()))


**7.5 Real patch analysis.** Files, hunks, added/deleted lines, file types, empty/malformed status; comparison with recorded metadata where it exists.

In [ ]:
if len(PATCH_DF) == 0:
    mark("real patch analysis", "NOT EXECUTED", "no agent.patch files found")
else:
    P = PATCH_DF
    print("patches: %d | empty: %d | malformed: %d | touch test files: %d" % (len(P), P.patch_empty.sum(), P.patch_malformed.sum(), P.patch_touches_tests.sum()))
    NE = P[~P.patch_empty]
    print("\nPatch size among non-empty patches (changed lines = added + deleted, parsed by this notebook):"); display(NE[["patch_files", "patch_hunks", "patch_added", "patch_deleted", "patch_changed_lines"]].describe().round(2))
    print("File extensions touched (patches containing each):"); ext = Counter(e for s in NE.patch_exts for e in s.split(";") if e); display(pd.Series(ext).sort_values(ascending=False).head(12).rename("patches").to_frame())
    # compare parsed structure with recorded metadata (agreement, not truth)
    if has(MERGED, "agent_files", "patch_files") or has(MERGED, "agent_lines", "patch_changed_lines"):
        M2 = MERGED[MERGED.has_patch_file == True].copy(); cmpd = {}
        if "agent_files" in M2:
            a = num(M2, "agent_files"); ok = a.notna(); cmpd["agent_files == parsed file count"] = float((a[ok] == M2.loc[ok, "patch_files"]).mean()) if ok.any() else None
        if "agent_lines" in M2:
            a = num(M2, "agent_lines"); ok = a.notna()
            if ok.any():
                cmpd["agent_lines == added+deleted"] = float((a[ok] == M2.loc[ok, "patch_changed_lines"]).mean())
                cmpd["agent_lines == added only"] = float((a[ok] == M2.loc[ok, "patch_added"]).mean())
        print("\nAgreement between result.json metadata and parsed patches (which definition `agent_lines` uses is INFERRED from these rates, not documented):", {k: (None if v is None else round(v, 3)) for k, v in cmpd.items()})
        RESULTS["patches"]["metadata_agreement"] = cmpd
    if has(MERGED, "resolved", "patch_changed_lines"):
        t = rate_table(MERGED[MERGED.has_patch_file == True].assign(size_bin=lambda d: pd.cut(d.patch_changed_lines, [-1, 0, 5, 20, 60, 10**9], labels=["0", "1-5", "6-20", "21-60", ">60"])), "size_bin")
        print("\nResolution by patch size (observational; patch size is confounded with task difficulty):"); display(t.round(3)); RESULTS["patches"]["resolution_by_size"] = t.astype({"size_bin": str}).to_dict("records")
    if has(MERGED, "resolved", "patch_touches_tests"):
        t = rate_table(MERGED[MERGED.has_patch_file == True], "patch_touches_tests"); print("Resolution by whether the patch touches test files:"); display(t.round(3))
    print("\nWhat patches CAN'T tell us: whether any edit caused a regression -- that needs a reproducible before/after test run, which this dataset does not contain.")
    RESULTS["patches"].update({"n": len(P), "empty": int(P.patch_empty.sum()), "malformed": int(P.patch_malformed.sum()), "touch_tests": int(P.patch_touches_tests.sum()),
                               "median_changed_lines_nonempty": float(NE.patch_changed_lines.median()) if len(NE) else None})
    P.drop(columns=["_patch_src"]).to_csv(TAB / "patch_analysis.csv", index=False)
    mark("real patch analysis", "EXECUTED", "%d patches" % len(P))


**7.6 Do the logs contain usable test output?** Scans `monitor.log` / `worker_*.log` for pytest-like lines and reports exact file:line provenance. Nothing found here is fed to the classifier or used to infer per-edit outcomes.

In [ ]:
PYTEST_SUMMARY = re.compile(r"\b\d+ (?:passed|failed|errors?)\b.*\bin \d+(?:\.\d+)?s")
FAIL_ID = re.compile(r"^(?:FAILED|ERROR) \S+::\S+")
TRACEBACK = re.compile(r"Traceback \(most recent call last\)")
PYTEST_CMD = re.compile(r"pytest")
LOGEV = []
if not LOG_FILES:
    mark("log test-evidence scan", "NOT EXECUTED", "no monitor.log / worker_*.log found")
else:
    task_ids = set(MERGED["task"].dropna().astype(str)) if len(MERGED) else set()
    lines_seen, examples = 0, []
    for v in LOG_FILES:
        c = Counter(); tagged = 0; n = 0
        for ln_no, line in enumerate(v.iter_lines(), 1):
            n += 1; lines_seen += 1
            if lines_seen > MAX_LOG_LINES: break
            hit = None
            if PYTEST_SUMMARY.search(line): hit = "pytest_summary"
            elif FAIL_ID.match(line): hit = "failed_id_line"
            elif TRACEBACK.search(line): hit = "traceback"
            elif PYTEST_CMD.search(line): hit = "mentions_pytest"
            if hit:
                c[hit] += 1
                if hit in ("pytest_summary", "failed_id_line") and any(t in line for t in task_ids if len(t) > 6): tagged += 1
                if len(examples) < 25 and hit in ("pytest_summary", "failed_id_line"): examples.append({"file": v.display, "line_no": ln_no, "kind": hit, "text": line[:200]})
        LOGEV.append({"log": v.display, "run": run_task_of(v)[0], "lines": n, **{k: c.get(k, 0) for k in ("pytest_summary", "failed_id_line", "traceback", "mentions_pytest")}, "summary_lines_with_task_id": tagged})
        if lines_seen > MAX_LOG_LINES: print("line cap reached; remaining logs not scanned"); break
    LOGDF = pd.DataFrame(LOGEV); LOGDF.to_csv(TAB / "log_test_evidence_inventory.csv", index=False)
    tot = LOGDF[["lines", "pytest_summary", "failed_id_line", "traceback", "mentions_pytest", "summary_lines_with_task_id"]].sum()
    print("log files scanned: %d | lines: %d" % (len(LOGDF), tot["lines"])); print(tot.to_string())
    print("\nExamples with exact provenance (file : line):"); display(pd.DataFrame(examples))
    usable = bool(tot["pytest_summary"] > 0)
    print("""
Interpretation: counts above are raw pattern matches. Even where pytest-like lines exist, they are run-level logs: a line is only usable as
before/after evidence if it can be tied to ONE task AND to a position before/after a specific edit. '%s' summary lines carry a task id on the same line.
This notebook therefore does NOT feed log text to the classifier and does NOT infer edit-level test outcomes from logs.""" % int(tot["summary_lines_with_task_id"]))
    RESULTS["logs"] = {"files": len(LOGDF), "lines": int(tot["lines"]), "pytest_summary_lines": int(tot["pytest_summary"]), "failed_id_lines": int(tot["failed_id_line"]),
                       "traceback_lines": int(tot["traceback"]), "summary_lines_with_task_id": int(tot["summary_lines_with_task_id"]),
                       "usable_edit_level_before_after_evidence": False, "examples": examples[:10]}
    mark("log test-evidence scan", "EXECUTED", "pytest-like lines present" if usable else "no pytest-like output found")


**7.7 Replay feasibility.** Checks for base-commit fields, repository snapshots and installed test suites. Replay is only attempted if all prerequisites exist; otherwise the reasons are recorded.

In [ ]:
FEAS = {"replay_attempted": False, "replay_feasible": False, "reasons": [], "repos": []}
if not len(RES_DF) or "repo" not in RES_DF:
    FEAS["reasons"].append("no result.json 'repo' field available")
else:
    keys = set()
    for v in RESULT_FILES[:200]:
        try: keys |= set(json.loads(v.read_text()).keys())
        except Exception: pass
    commit_keys = sorted(k for k in keys if re.search(r"commit|sha|base_ref|revision", k, re.I))
    clones = sorted({"/".join(v.parts[:4]) for v in DS_FILES if ".git" in v.parts})
    for repo in sorted(RES_DF["repo"].dropna().astype(str).unique()):
        mod = repo.split("/")[-1].replace("-", "_").lower()
        try: spec = importlib.util.find_spec(mod)
        except Exception: spec = None
        ver = None
        if spec:
            try:
                import importlib.metadata as md; ver = md.version(repo.split("/")[-1])
            except Exception: ver = "unknown"
        loc = list(spec.submodule_search_locations or []) if spec and spec.submodule_search_locations else []
        tests_shipped = any(os.path.isdir(os.path.join(l, "tests")) or os.path.isdir(os.path.join(os.path.dirname(l), "tests")) for l in loc)
        FEAS["repos"].append({"repo": repo, "tasks": int((RES_DF["repo"] == repo).sum()), "installed_module": bool(spec), "installed_version": ver, "installed_tests_dir_found": tests_shipped})
    FEAS["base_commit_fields_in_result_json"] = commit_keys; FEAS["local_git_clones_in_dataset"] = clones
    if not commit_keys: FEAS["reasons"].append("result.json has no base-commit/revision field, so the repository state each patch applies to is unknown")
    if not clones: FEAS["reasons"].append("the dataset contains no repository snapshots/clones; only agent.patch + result.json")
    if not any(r["installed_tests_dir_found"] for r in FEAS["repos"]): FEAS["reasons"].append("no installed copy of the target repositories ships a tests/ directory matching the task's test files")
    FEAS["reasons"].append("the project forbids external downloads/API calls here, so cloning repositories at a base commit is not performed")
    FEAS["reasons"].append("`base_fails`/`gold_passes` are recorded values whose semantics are unverified; they are not per-edit test outputs")
    display(pd.DataFrame(FEAS["repos"]))
ready = bool(FEAS.get("base_commit_fields_in_result_json")) and bool(FEAS.get("local_git_clones_in_dataset")) and any(r["installed_tests_dir_found"] for r in FEAS["repos"])
FEAS["conclusion"] = ("Prerequisites appear present, but replay is NOT attempted by this notebook (needs per-task dependency setup)." if ready
                      else "Replay of Gemma patches is NOT feasible with the available evidence and was NOT attempted.")
for r in FEAS["reasons"]: print(" -", r)
print(FEAS["conclusion"])
RESULTS["replay_feasibility"] = FEAS
mark("Gemma patch replay", "NOT EXECUTED", ("infeasible without repo snapshots/base commits/test environments (see reasons)" if len(RES_DF) else "dataset not loaded"))


## 8. Figures
Every figure has a title, axis labels and a printed explanation of what it does and does not show.

In [ ]:
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / name, dpi=130, bbox_inches="tight"); plt.show()

def heat(ax, cm, title):
    cm = cm.loc[cm.sum(axis=1) > 0, :]
    cols = [c for c in cm.columns if cm[c].sum() > 0 or c in cm.index]
    cm = cm[cols]
    if HAVE_SNS: sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    else:
        ax.imshow(cm.values, cmap="Blues"); ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=90); ax.set_yticks(range(len(cm))); ax.set_yticklabels(cm.index)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]): ax.text(j, i, int(cm.values[i, j]), ha="center", va="center")
    ax.set_title(title, fontsize=10); ax.set_xlabel("predicted label"); ax.set_ylabel("expected label (authored)")
    ax.tick_params(axis="x", rotation=45)

# 1. confusion matrices, original vs improved, agent protocol, tier 1+2
sel = PRED[(PRED.tier == 1) & (PRED["mode"] == "x")]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
reps = {}
for ax, m in zip(axes, ["B_orig", "C_v2"]):
    d = sel[sel.method == m]; reps[m] = label_report(d.truth, d.pred)
    heat(ax, reps[m]["confusion"], "%s  (n=%d, accuracy=%.2f)" % (m, reps[m]["n"], reps[m]["accuracy"]))
fig.suptitle("Confusion matrices, PRIMARY set: %d tier-1 (oracle-confirmed) scenarios, pytest -x protocol" % reps["B_orig"]["n"])
savefig("confusion_matrices.png")
print("Explanation: rows are the authored expected labels, columns the classifier's output. Off-diagonal cells are errors. These are hand-built scenarios,\n"
      "and v2 was tuned on part of them (dev scenarios), so the v2 matrix describes performance on these constructed scenarios only; see the holdout rows.")

# 2. per-class F1
pc = pd.concat([reps[m]["per_class"].assign(method=m) for m in reps])
pc = pc[pc.support > 0]
fig, ax = plt.subplots(figsize=(9, 4))
piv = pc.pivot(index="label", columns="method", values="f1")
piv.plot.bar(ax=ax); ax.set_ylim(0, 1.05); ax.set_title("Per-class F1 (labels with support > 0), tier-1 scenarios, pytest -x protocol"); ax.set_xlabel("expected label"); ax.set_ylabel("F1 (0-1)")
for lbl in ax.get_xticklabels(): lbl.set_rotation(30); lbl.set_ha("right")
savefig("per_class_f1.png")
print("Explanation: F1 per label. Support per class is 1-8 cases, so a single scenario moves a bar a lot; treat as descriptive only.\n" + pc[["method", "label", "support", "precision", "recall", "f1"]].round(2).to_string(index=False))

# 3. error types
fig, ax = plt.subplots(figsize=(9, 4))
ETp = ET.copy(); ETp.index = ["%s/%s/t%s" % i for i in ETp.index]
if len(ETp): ETp.plot.bar(stacked=True, ax=ax)
ax.set_title("Mistakes by type and tier (tier 1 = oracle-confirmed, tier 2 = authored-only)"); ax.set_xlabel("method / output mode / tier"); ax.set_ylabel("number of scenarios")
for lbl in ax.get_xticklabels(): lbl.set_rotation(30); lbl.set_ha("right")
savefig("error_types.png")
print("Explanation: counts of wrong labels by kind. 'false regression' means the classifier would tell an agent to inspect/revert a patch that did not break anything.")

# 4-6. Gemma plots
if len(MERGED) and "resolved" in MERGED and "run" in MERGED:
    pr = PER_RUN.sort_values("resolution_rate")
    fig, ax = plt.subplots(figsize=(8, max(4, 0.3 * len(pr))))
    ax.barh(pr["run"].astype(str), pr["resolution_rate"], xerr=[pr["resolution_rate"] - pr["ci_lo"], pr["ci_hi"] - pr["resolution_rate"]], color="#4c72b0", capsize=2)
    ax.set_title("Resolution rate per run (Wilson 95% CI), recorded `resolved` flag"); ax.set_xlabel("fraction of attempted tasks resolved"); ax.set_ylabel("run")
    savefig("gemma_resolution_per_run.png")
    print("Explanation: runs attempted different numbers/subsets of tasks, so bars are not strictly comparable; see the common-task table. Differences may reflect prompt, seed, subset or noise -- no causal claim.")
    if "tool_calls" in MERGED and MERGED.tool_calls.notna().any():
        fig, ax = plt.subplots(figsize=(8, 4))
        for val, lab in ((1.0, "resolved"), (0.0, "unresolved")):
            ax.hist(MERGED.loc[MERGED.resolved == val, "tool_calls"].dropna(), bins=30, alpha=0.55, label=lab)
        ax.set_title("Distribution of tool calls per task-run, by resolution outcome"); ax.set_xlabel("tool_calls (recorded)"); ax.set_ylabel("number of task-runs"); ax.legend()
        savefig("gemma_tool_calls.png")
        print("Explanation: overlap of the two histograms shows tool-call count alone does not separate outcomes; the plot does not say why.")
    if "patch_changed_lines" in MERGED and MERGED.patch_changed_lines.notna().any():
        d = MERGED.dropna(subset=["patch_changed_lines"]); d = d[d.patch_changed_lines > 0]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(np.log10(d.patch_changed_lines), bins=30); ax.set_title("Distribution of agent.patch size (non-empty patches, log10 scale)"); ax.set_xlabel("log10(added + deleted lines), parsed from agent.patch"); ax.set_ylabel("number of patches")
        savefig("gemma_patch_size.png")
        print("Explanation: most patches are small if mass is left of 2 (100 lines). This describes the data only.")
mark("figures", "EXECUTED", "saved to %s" % FIG)


## 9. Exports and integrity checks
Verifies the original sources are unchanged, writes `results.json` and `merged_dataset.csv`, and round-trips the JSON.

In [ ]:
class NpEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return None if np.isnan(o) else float(o)
        if isinstance(o, (np.bool_,)): return bool(o)
        if isinstance(o, (set, tuple)): return list(o)
        if isinstance(o, Path): return str(o)
        return str(o)

# ---- integrity checks before writing anything
for fname, h in ORIG_HASHES.items():
    assert sha((SRC / fname).read_text(encoding="utf-8")) == h, "ORIGINAL SOURCE MODIFIED DURING RUN: " + fname
print("integrity: original classifier/bench/snippet unchanged since load (sha256 verified)")
assert len(PRED) == len(EXEC) * 2 * len(METHODS)
assert PRED.groupby(["name", "mode"]).truth.nunique().max() == 1

RESULTS["controlled_suite"] = {"n_dev": len(CT_DEV), "n_holdout": len(CT_HOLDOUT), "tier_counts": INST["tier"].value_counts().sort_index().to_dict(),
    "design": "purpose-built multi-module repos, real git + real pytest; ground truth = authored label cross-checked by an outcome-level oracle from full JUnit runs",
    "scenarios": INST.to_dict("records")}
def headline(set_name, mode, method, key):
    r = CMP[(CMP.case_set == set_name) & (CMP["mode"] == mode) & (CMP.method == method)]
    return None if r.empty else r.iloc[0][key]
RESULTS["meta"]["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
with open(OUT / "results.json", "w") as f: json.dump(RESULTS, f, indent=2, cls=NpEncoder)
json.load(open(OUT / "results.json"))   # round-trip check
if len(MERGED):
    MERGED.to_csv(OUT / "merged_dataset.csv", index=False); print("wrote merged_dataset.csv:", MERGED.shape)
else:
    print("merged_dataset.csv NOT written: no Gemma data was loaded")
print("wrote results.json")


## 10. Auto-generated research report
Writes `report.md` (Paper-Track-oriented) with synthetic, controlled and Gemma findings kept in separate sections, limitations, and a conclusion computed strictly from this run's numbers.

In [ ]:
def md_table(df, floatfmt=3):
    if df is None or len(df) == 0: return "_(none)_\n"
    d = df.copy()
    for c in d.columns:
        d[c] = d[c].map(lambda x: "" if (isinstance(x, float) and pd.isna(x)) else (("%.*f" % (floatfmt, x)) if isinstance(x, float) else x))
    d = d.fillna("").astype(str).apply(lambda col: col.str.replace("|", "/", regex=False).str.replace("\n", " ", regex=False))
    return "| " + " | ".join(d.columns) + " |\n|" + "---|" * len(d.columns) + "\n" + "\n".join("| " + " | ".join(r) + " |" for r in d.values.tolist()) + "\n"

g = lambda s, m, me, k: headline(s, m, me, k)
P1, P2 = "PRIMARY: tier 1 (oracle-confirmed)", "tier 2 only (authored-only, NOT oracle-confirmed)"
n12 = g(P1, "x", "B_orig", "n")
acc = lambda s, m, me: g(s, m, me, "label_accuracy")
fmt = lambda x: "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else "%.2f" % x
EXEC_TABLE = pd.DataFrame([{"experiment": k, "status": v["status"], "note": v["reason"]} for k, v in RESULTS["executed"].items()])
gem_exec = RESULTS["executed"].get("Gemma exploratory analysis", {}).get("status") == "EXECUTED"

L = []
A = L.append
A("# Failure-attribution-guided agentic learning: experimental report (auto-generated)\n")
A("Generated %s by the notebook; every number below was computed in the run that produced this file.\n" % RESULTS["meta"]["finished"])
A("**Research question.** Can a coding agent improve debugging/recovery decisions by distinguishing pre-existing test failures from regressions introduced by its own edits?\n")
A("**Bottom line (read first).** This notebook evaluates the *classifier* on hand-built scenarios. It does **not** and **cannot** answer the research question about agent behaviour: the Gemma run data contains no per-edit before/after test outputs, and no agent was run with versus without attribution.\n")
A("## 0. Experiments that executed\n"); A(md_table(EXEC_TABLE))
A("## 1. Evidence labelling used throughout\n- **Observed**: values recorded in the CSV / result.json / patch / test runs executed here.\n- **Derived**: computed from observed values (rates, CIs, parsed patch sizes, oracle labels).\n- **Heuristic**: classifier labels, run groups, inferred field semantics.\n- **Unavailable**: per-edit test output for Gemma runs; base commits; repository snapshots.\n")
A("## 2. Synthetic benchmark (original attribution_bench.py)\n")
A("Source of labels: the author of the scenarios (not independent). Preliminary functional validation only.\n")
A("- Original classifier, original protocol, authored labels: **%d/%d** correct (accuracy %.2f, Wilson 95%% CI %.2f-%.2f)." % (round(RESULTS["synthetic_bench"]["orig10_accuracy"] * RESULTS["synthetic_bench"]["orig10_n"]), RESULTS["synthetic_bench"]["orig10_n"], RESULTS["synthetic_bench"]["orig10_accuracy"], *RESULTS["synthetic_bench"]["orig10_ci95"]))
A("- Excluding the questionable `logic_progress` case: accuracy %.2f on %d cases." % (RESULTS["synthetic_bench"]["orig9_accuracy"], RESULTS["synthetic_bench"]["orig9_n"]))
A("- `logic_progress` audit: %s." % RESULTS["synthetic_bench"]["logic_progress_audit"])
A("- Oracle confirmation of authored labels (tier 1 confirmed / 2 authored-only / 3 questionable): %s." % RESULTS["synthetic_bench"]["orig10_oracle_tiers"])
A("- Harness equivalence with original run_case(): all predictions identical (checked).\n")
A("Per-class (original classifier, orig10):\n"); A(md_table(pd.DataFrame(RESULTS["synthetic_bench"]["orig10_per_class"]).query("support > 0 or predicted > 0")))
A("## 3. Controlled experiment (purpose-built repos, real git + pytest)\n")
A("Design: %s. Scenario counts: dev=%d, holdout=%d; tiers %s. **Primary result = tier 1 (oracle-confirmed). Tier 2 (author-assigned labels only) is reported separately and is not pooled into headline numbers. Not real third-party repositories and not Gemma performance.**\n" % (RESULTS["controlled_suite"]["design"], len(CT_DEV), len(CT_HOLDOUT), RESULTS["controlled_suite"]["tier_counts"]))
for sname in (P1, P2, "ct_holdout tier 1", "tier 3 only (ambiguous/questionable)"):
    for mode in ("x", "full"):
        sub_ = CMP[(CMP.case_set == sname) & (CMP["mode"] == mode)].drop(columns=["case_set", "mode"])
        A("**%s -- output mode %s**\n" % (sname, "-x (agent protocol)" if mode == "x" else "full run")); A(md_table(sub_, 2))
A("## 4. Classifier review\n"); A("Findings and whether a real execution reproduced them:\n"); A(md_table(FIND)); A("Not tested here:\n"); A("\n".join("- " + t for t in RESULTS["classifier_review"]["not_tested_findings"]) + "\n")
A("Change log of the improved variant (original preserved separately):\n"); A(md_table(CHANGE_LOG))
A("Remaining weaknesses of the improved variant (each verified by a scenario where one exists):\n"); A(md_table(V2_WEAK))
A("Mistakes on non-ambiguous cases (tier 1 = oracle-confirmed; tier 2 = authored-only):\n"); A(md_table(ERR[["method", "mode", "name", "tier", "truth", "pred", "conf", "error_type"]]))
A("## 5. Real Gemma-run exploratory findings (observational only)\n")
if gem_exec:
    A("Loaded %d task-run rows over %d runs / %d tasks. Data-quality summary: `%s`.\n" % (RESULTS["gemma"]["eda"]["n_rows"], RESULTS["gemma"]["eda"]["n_runs"], RESULTS["gemma"]["eda"]["n_tasks"], json.dumps(RESULTS["gemma"]["quality"], cls=NpEncoder)))
    A("Per-run resolution (observed counts; derived rates):\n"); A(md_table(PER_RUN[["run", "n_tasks", "resolved_n", "resolution_rate", "ci_lo", "ci_hi"]], 3))
    A("Group definition: %s. Associations with `resolved` (Spearman, correlation only):\n" % RESULTS["gemma"]["eda"]["group_definition"]); A(md_table(REL, 3))
    A("These associations are confounded by task difficulty, run and category and imply no causal effect.\n")
else:
    A("**Not executed**: " + RESULTS["executed"].get("Gemma exploratory analysis", RESULTS["executed"].get("Gemma dataset discovery", {"reason": "dataset not available"}))["reason"] + "\n")
A("### Patches\n"); A("%s\n" % json.dumps(RESULTS["patches"], cls=NpEncoder) if RESULTS["patches"] else "No patch analysis (no patches found).\n")
A("### Logs\n"); A("%s\n" % json.dumps({k: v for k, v in RESULTS["logs"].items() if k != "examples"}, cls=NpEncoder) if RESULTS["logs"] else "No log scan.\n")
A("### Replay feasibility\n"); A("%s\n" % RESULTS["replay_feasibility"].get("conclusion", "not assessed")); A("\n".join("- " + r for r in RESULTS["replay_feasibility"].get("reasons", [])) + "\n")
LIMITS = [
 "All classifier-evaluation scenarios are hand-written by the classifier's author; labels, scenario selection and the oracle share that bias.",
 "The outcome-level oracle is independent of the classifier's text parsing but cannot see progress inside a still-failing test; those cases are tier 2.",
 "v2 was developed on the dev scenarios; its dev scores describe constructed scenarios only. The holdout is small (n=%d) and same-author." % len(CT_HOLDOUT),
 "v2 has verified remaining weaknesses: dotted missing modules are assumed repo-local (W1); a patch that edits tests can still yield PASSED (W2); -x handling relies on a heuristic tuned on dev scenarios (W3).",
 "Tier-2 labels are author-assigned only and are not independently verified.",
 "Controlled repos are small and synthetic; no third-party repository with a reproducible base commit was available offline.",
 "Sample sizes are tiny; Wilson intervals are wide and per-class F1 rests on 1-8 cases.",
 "Confidence values cannot be calibrated at this sample size.",
 "D_prompt_rules is an emulation of the written rules, not measured LLM behaviour; no agent or LLM was run.",
 "Gemma data has no per-edit before/after test output; unresolved != regression, base failure != environment failure, and `gold_passes`/`base_fails` semantics are unverified.",
 "Gemma comparisons across runs/prompt variants are observational, confounded by task subsets, seeds and configuration; run-group definitions are heuristic.",
 "Flaky/nondeterministic tests and repeated-baseline noise were not evaluated.",
]
RESULTS["limitations"] = LIMITS
A("## 6. Limitations\n"); A("\n".join("- " + x for x in LIMITS) + "\n")
ii = lambda x: "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else str(int(x))
b_fr, c_fr = g(P1, "x", "B_orig", "false_regression"), g(P1, "x", "C_v2", "false_regression")
b_mr, c_mr = g(P1, "x", "B_orig", "missed_regression"), g(P1, "x", "C_v2", "missed_regression")
b_fr_f, c_fr_f = g(P1, "full", "B_orig", "false_regression"), g(P1, "full", "C_v2", "false_regression")
repro = [r["id"] for r in RESULTS["classifier_review"]["findings_reproduction"] if str(r["reproduced_by_orig"]).startswith("YES")]
notrepro = [r["id"] for r in RESULTS["classifier_review"]["findings_reproduction"] if not str(r["reproduced_by_orig"]).startswith("YES")]
A("## 7. Conclusion (strictly from observed results)\n")
A("1. **Primary result (tier 1: authored label confirmed by the outcome-level oracle, n=%s, pytest -x protocol):** the original classifier matched the expected label in %s of cases and the improved variant (v2) in %s. These describe performance on *constructed scenarios*, not general reliability; v2 was developed on most of them (development contamination), and the labelled ground truth is only as independent as the oracle." % (n12, fmt(acc(P1, "x", "B_orig")), fmt(acc(P1, "x", "C_v2"))))
A("2. **Tier 2 (author-assigned labels only, n=%s) is reported separately:** original %s, v2 %s. These labels are NOT independently verified and are never pooled into the primary result." % (g(P2, "x", "B_orig", "n"), fmt(acc(P2, "x", "B_orig")), fmt(acc(P2, "x", "C_v2"))))
A("3. **The -x protocol is itself a limitation.** On tier 1 under -x, false-regression decisions were %s (original) vs %s (v2), missed regressions %s vs %s; without -x (full baseline) false regressions were %s vs %s. With a truncated baseline a failure that existed but was never reached can look new, and v2's linkage heuristic only partly mitigates this on development scenarios; it does not solve the problem in general. A full baseline run without -x gives stronger attribution evidence and is the recommended protocol." % (ii(b_fr), ii(c_fr), ii(b_mr), ii(c_mr), ii(b_fr_f), ii(c_fr_f)))
A("4. **v2 still has verified weaknesses** (see the weaknesses table): %s." % "; ".join("%s %s" % (r["id"], "confirmed" if r["confirmed"] else "not confirmed") for r in RESULTS["classifier_review"]["v2_remaining_weaknesses"]))
A("5. Findings about the ORIGINAL classifier reproduced by real executions: %s. Not reproduced: %s." % (", ".join(repro) or "none", ", ".join(notrepro) or "none"))
_f4 = PRED[(PRED.method == "B_orig") & (PRED["mode"] == "x") & PRED.name.isin(["ct_noisy_pass", "ct_noisy_pass_own_line"])].set_index("name")["pred"].to_dict()
if _f4.get("ct_noisy_pass") == "PASSED" and _f4.get("ct_noisy_pass_own_line") not in (None, "PASSED"):
    A("   F4 detail: reproduced only when the exception-looking stdout line starts at column 0.")
A("6. Labels express evidence *consistent with* an interpretation (for example, a regression associated with the patch). Traceback overlap and changed error text are clues, not proof of causation.")
A("7. **The research question is not answered by this evidence.** Neither the constructed scenarios nor the Gemma traces (no per-edit test evidence, no with/without-attribution comparison) can show that attribution improves agent recovery.")
REPORT = "\n".join(L)
(OUT / "report.md").write_text(REPORT, encoding="utf-8")
(OUT / "results.json").write_text(json.dumps(RESULTS, indent=2, cls=NpEncoder), encoding="utf-8")
print(REPORT[:6000]); print("\n... [report.md: %d chars written to %s]" % (len(REPORT), OUT / "report.md"))
mark("report + exports", "EXECUTED")


## 11. Output inventory and what actually executed

In [ ]:
print("OUTPUT FILES under", OUT.resolve())
for p in sorted(OUT.rglob("*")):
    if p.is_file() and "src/__pycache__" not in str(p):
        print("  %-60s %8d bytes" % (p.relative_to(OUT), p.stat().st_size))
required = ["results.json", "report.md"] + (["merged_dataset.csv"] if len(MERGED) else [])
for r in required: assert (OUT / r).exists() and (OUT / r).stat().st_size > 0, "missing output " + r
print("\nWHAT ACTUALLY EXECUTED:")
display(pd.DataFrame([{"experiment": k, **v} for k, v in RESULTS["executed"].items()]))
